# Evaluating Large Language Models for Automated BioCRNpyler Script Generation

This notebook evaluates the capabilities of various Large Language Models (including Gemini NIST version, Claude, and ChatGPT) to generate functional BioCRNpyler Python scripts for building Chemical Reaction Network (CRN) models.

**Target Model:** The AI models were tasked with building a Chemical Reaction Network (CRN) representing the DNA strand displacement **AND gate** detailed in *Programming Cell-free Biosensors with DNA Strand Displacement Circuits* ([Jung et al., Nature Chemical Biology 2022](https://www.nature.com/articles/s41589-021-00962-9)).

To rigorously assess each model, the evaluation is divided into two main sections: **Zero-Shot Testing**. Within each section, the results are organized by the AI tool used, followed by the individual runs (attempts). 

To ensure an objective evaluation of each model's coding accuracy and understanding of the BioCRNpyler framework, the testing followed a strict procedure:
* **Raw Output Execution:** The code generated by each AI was copied and pasted directly into individual notebook cells without human modification or debuggingwith the sole exception of standardizing the output filenames.
* **Sequential Running:** All code blocks within a specific test run were executed in sequential order. 
* **Error Retention:** If any block of code failed to compile or run, the resulting traceback and error messages were retained in the output, and execution naturally continued to the next designated block.

*Note: Gemini generation was performed using the NIST version of Gemini on the [Pro] option. Claude and ChatGPT script generation was performed using the Free Plan.*

In [1]:
from pathlib import Path
# Get base directory for BioCRNpyler scripts
directory = Path.cwd()
for p in [directory, *directory.parents]:
    if p.name == 'TMSE_BioCRNpyler_Library':
        base_dir = p
        break

# Directory for models
model_dir = (base_dir / 'models' / 'AI_generated models').as_posix()

## Zero-Shot Testing

The first section contains the results for the zero-shot test, where the models were asked to generate the AND gate script without any prior examples or context regarding `BioCRNpyler` syntax. 

Below is the exact prompt used for the zero-shot test.

> **Prompt:** Translate the following chemical reactions and kinetic parameters into a complete BioCRNpyler Python script to build the corresponding Chemical Reaction Network (CRN). 
>
> ### tetO-regulated Template
>
> TetR + D_tetO <---(k_rep,T, k_derep,T)---> TetRD
>
> TetR + tet <---(k_ind,T, k_unin,T)---> TetR-tet
>
> RNAP + D_tetO <---(k_bind, k_unbind)---> RD_tetO
>
> RD_tetO --(k_m,tetO)--> InvR_T + RNAP + D_tetO
>
> ### smtO-Regulated Template
>
> SmtB + D_smtO <---(k_rep,S, k_derep,S)---> SmtBD
>
> SmtB + Zn <---(k_ind,S, k_unin,S)---> SmtB-Zn
>
> RNAP + D_smtO <---(k_bind, k_unbind)---> RD_smtO
>
> RD_smtO --(k_m,smtO)--> InvR_S + RNAP + D_smtO
>
> ### Strand Displacements
>
> InvR_T + InvR_S + AND-Gate --(k_SD,AND)--> InvR_D + Gated_TS
>
> InvR_D + RQ --(k_SD)--> SD + Q
>
> ### Parameters
>
> k_m,tetO = 0.1 /sec
>
> k_m,smtO = 0.04 /sec
>
> k_bind = 56 /uM-sec
>
> k_unbind = 0.2 /sec
>
> k_SD = 0.04 /uM-sec
>
> k_SD-AND = 0.08 /uM-sec
>
> k_rep,T = 2.98 /uM-sec
>
> k_derep,T = 0.001 /uM-sec
>
> k_ind,T = 4.0 /uM-sec
>
> k_unin,T = 0.022 /sec
>
> k_rep,S = 3.0 /uM-sec
>
> k_derep,S = 0.18 /uM-sec
>
> k_ind,S = 80 /uM-sec
>
> k_unin,S = 0.1 /sec


## Gemini [Pro] Attempts [6/10 Successful]

### Attempt 1 [Failed]

In [2]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Define Species
# ==========================================

# Transcription Factors & Inducers
TetR = Species("TetR")
tet = Species("tet")
SmtB = Species("SmtB")
Zn = Species("Zn")
RNAP = Species("RNAP")

# DNA Templates
D_tetO = Species("D_tetO")
D_smtO = Species("D_smtO")

# Protein-DNA and Protein-Inducer Complexes
TetRD = Species("TetRD")
TetR_tet = Species("TetR_tet")
SmtBD = Species("SmtBD")
SmtB_Zn = Species("SmtB_Zn")
RD_tetO = Species("RD_tetO")
RD_smtO = Species("RD_smtO")

# RNA Transcripts and Strand Displacement Components
InvR_T = Species("InvR_T")
InvR_S = Species("InvR_S")
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# Combine all species into a list
species_list = [
    TetR, tet, SmtB, Zn, RNAP, 
    D_tetO, D_smtO, 
    TetRD, TetR_tet, SmtBD, SmtB_Zn, RD_tetO, RD_smtO, 
    InvR_T, InvR_S, AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
]

# ==========================================
# 2. Define Kinetic Parameters (Units: uM, sec)
# ==========================================
k_m_tetO = 0.1
k_m_smtO = 0.04
k_bind = 56.0
k_unbind = 0.2
k_SD = 0.04
k_SD_AND = 0.08

k_rep_T = 2.98
k_derep_T = 0.001
k_ind_T = 4.0
k_unin_T = 0.022

k_rep_S = 3.0
k_derep_S = 0.18
k_ind_S = 80.0
k_unin_S = 0.1

# ==========================================
# 3. Define Reactions
# ==========================================
reactions = []

# --- tetO-regulated Template ---
# TetR + D_tetO <---> TetRD
reactions.append(Reaction(inputs=[TetR, D_tetO], outputs=[TetRD], k=k_rep_T, k_rev=k_derep_T))

# TetR + tet <---> TetR-tet
reactions.append(Reaction(inputs=[TetR, tet], outputs=[TetR_tet], k=k_ind_T, k_rev=k_unin_T))

# RNAP + D_tetO <---> RD_tetO
reactions.append(Reaction(inputs=[RNAP, D_tetO], outputs=[RD_tetO], k=k_bind, k_rev=k_unbind))

# RD_tetO ---> InvR_T + RNAP + D_tetO
reactions.append(Reaction(inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO], k=k_m_tetO))


# --- smtO-Regulated Template ---
# SmtB + D_smtO <---> SmtBD
reactions.append(Reaction(inputs=[SmtB, D_smtO], outputs=[SmtBD], k=k_rep_S, k_rev=k_derep_S))

# SmtB + Zn <---> SmtB-Zn
reactions.append(Reaction(inputs=[SmtB, Zn], outputs=[SmtB_Zn], k=k_ind_S, k_rev=k_unin_S))

# RNAP + D_smtO <---> RD_smtO
reactions.append(Reaction(inputs=[RNAP, D_smtO], outputs=[RD_smtO], k=k_bind, k_rev=k_unbind))

# RD_smtO ---> InvR_S + RNAP + D_smtO
reactions.append(Reaction(inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO], k=k_m_smtO))


# --- Strand Displacements ---
# InvR_T + InvR_S + AND-Gate ---> InvR_D + Gated_TS
reactions.append(Reaction(inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS], k=k_SD_AND))

# InvR_D + RQ ---> SD + Q
reactions.append(Reaction(inputs=[InvR_D, RQ], outputs=[SD, Q], k=k_SD))


# ==========================================
# 4. Compile the Chemical Reaction Network
# ==========================================
CRN = ChemicalReactionNetwork(species=species_list, reactions=reactions)

# Print the resulting CRN to verify structure and rates
print(CRN)

CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r1.xml') # export manually added

TypeError: Reaction.__init__() got an unexpected keyword argument 'k'

### Attempt 2 [Successful]

In [3]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

def build_crn():
    # ==========================================
    # 1. Define the Species
    # ==========================================
    
    # Global Species
    RNAP = Species("RNAP")
    
    # tetO-regulated Template Species
    TetR     = Species("TetR")
    D_tetO   = Species("D_tetO")
    TetRD    = Species("TetRD")
    tet      = Species("tet")
    TetR_tet = Species("TetR_tet")  # Hyphen replaced with underscore for valid Python variable
    RD_tetO  = Species("RD_tetO")
    InvR_T   = Species("InvR_T")
    
    # smtO-Regulated Template Species
    SmtB     = Species("SmtB")
    D_smtO   = Species("D_smtO")
    SmtBD    = Species("SmtBD")
    Zn       = Species("Zn")
    SmtB_Zn  = Species("SmtB_Zn")
    RD_smtO  = Species("RD_smtO")
    InvR_S   = Species("InvR_S")
    
    # Strand Displacements Species
    AND_Gate = Species("AND_Gate") 
    InvR_D   = Species("InvR_D")
    Gated_TS = Species("Gated_TS")
    RQ       = Species("RQ")
    SD       = Species("SD")
    Q        = Species("Q")
    
    # Combine into a single list
    species_list = [
        RNAP, TetR, D_tetO, TetRD, tet, TetR_tet, RD_tetO, InvR_T,
        SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
        AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
    ]
    
    # ==========================================
    # 2. Define Parameters 
    # ==========================================
    
    # Transcription / Binding Rates
    k_m_tetO = 0.1
    k_m_smtO = 0.04
    k_bind   = 56
    k_unbind = 0.2
    
    # Strand Displacement Rates
    k_SD     = 0.04
    k_SD_AND = 0.08
    
    # tetO Repression / Induction Rates
    k_rep_T   = 2.98
    k_derep_T = 0.001
    k_ind_T   = 4.0
    k_unin_T  = 0.022
    
    # smtO Repression / Induction Rates
    k_rep_S   = 3.0
    k_derep_S = 0.18
    k_ind_S   = 80
    k_unin_S  = 0.1

    # ==========================================
    # 3. Build the Reactions
    # ==========================================
    reactions = []
    
    # --- tetO-regulated Template ---
    # TetR + D_tetO <---> TetRD
    reactions.append(Reaction.from_massaction([TetR, D_tetO], [TetRD], k_forward=k_rep_T, k_reverse=k_derep_T))
    
    # TetR + tet <---> TetR-tet
    reactions.append(Reaction.from_massaction([TetR, tet], [TetR_tet], k_forward=k_ind_T, k_reverse=k_unin_T))
    
    # RNAP + D_tetO <---> RD_tetO
    reactions.append(Reaction.from_massaction([RNAP, D_tetO], [RD_tetO], k_forward=k_bind, k_reverse=k_unbind))
    
    # RD_tetO ---> InvR_T + RNAP + D_tetO
    reactions.append(Reaction.from_massaction([RD_tetO], [InvR_T, RNAP, D_tetO], k_forward=k_m_tetO))
    
    
    # --- smtO-Regulated Template ---
    # SmtB + D_smtO <---> SmtBD
    reactions.append(Reaction.from_massaction([SmtB, D_smtO], [SmtBD], k_forward=k_rep_S, k_reverse=k_derep_S))
    
    # SmtB + Zn <---> SmtB-Zn
    reactions.append(Reaction.from_massaction([SmtB, Zn], [SmtB_Zn], k_forward=k_ind_S, k_reverse=k_unin_S))
    
    # RNAP + D_smtO <---> RD_smtO
    reactions.append(Reaction.from_massaction([RNAP, D_smtO], [RD_smtO], k_forward=k_bind, k_reverse=k_unbind))
    
    # RD_smtO ---> InvR_S + RNAP + D_smtO
    reactions.append(Reaction.from_massaction([RD_smtO], [InvR_S, RNAP, D_smtO], k_forward=k_m_smtO))
    
    
    # --- Strand Displacements ---
    # InvR_T + InvR_S + AND-Gate ---> InvR_D + Gated_TS
    reactions.append(Reaction.from_massaction([InvR_T, InvR_S, AND_Gate], [InvR_D, Gated_TS], k_forward=k_SD_AND))
    
    # InvR_D + RQ ---> SD + Q
    reactions.append(Reaction.from_massaction([InvR_D, RQ], [SD, Q], k_forward=k_SD))

    # ==========================================
    # 4. Compile the CRN
    # ==========================================
    crn = ChemicalReactionNetwork(species=species_list, reactions=reactions)
    return crn

if __name__ == "__main__":
    my_crn = build_crn()
    
    # Print a summary of the network
    print(my_crn.pretty_print())

my_crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r2.xml')

Species(N = 21) = {
tet (@ 0),  Zn (@ 0),  TetR_tet (@ 0),  TetRD (@ 0),  TetR (@ 0),  SmtB_Zn (@ 0),  SmtBD (@ 0),  SmtB (@ 0),  SD (@ 0),  RQ (@ 0),  RNAP (@ 0),  RD_tetO (@ 0),  RD_smtO (@ 0),  Q (@ 0),  InvR_T (@ 0),  InvR_S (@ 0),  InvR_D (@ 0),  Gated_TS (@ 0),  D_tetO (@ 0),  D_smtO (@ 0),  AND_Gate (@ 0),  
}

Reactions (10) = [
0. TetR+D_tetO <--> TetRD
 Kf=k_forward * TetR * D_tetO
 Kr=k_reverse * TetRD
  k_forward=2.98
  k_reverse=0.001

1. TetR+tet <--> TetR_tet
 Kf=k_forward * TetR * tet
 Kr=k_reverse * TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. RNAP+D_tetO <--> RD_tetO
 Kf=k_forward * RNAP * D_tetO
 Kr=k_reverse * RD_tetO
  k_forward=56
  k_reverse=0.2

3. RD_tetO --> InvR_T+RNAP+D_tetO
 Kf=k_forward * RD_tetO
  k_forward=0.1

4. SmtB+D_smtO <--> SmtBD
 Kf=k_forward * SmtB * D_smtO
 Kr=k_reverse * SmtBD
  k_forward=3.0
  k_reverse=0.18

5. SmtB+Zn <--> SmtB_Zn
 Kf=k_forward * SmtB * Zn
 Kr=k_reverse * SmtB_Zn
  k_forward=80
  k_reverse=0.1

6. RNAP+D_smtO <--> RD_smtO

True

### Attempt 3 [Failed]

In [4]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Kinetic Parameters
# ==========================================
# Catalytic / Transcription rates (/sec)
k_m_tetO = 0.1
k_m_smtO = 0.04

# RNAP binding/unbinding (/uM-sec, /sec)
k_bind = 56.0
k_unbind = 0.2

# Strand Displacement rates (/uM-sec)
k_SD = 0.04
k_SD_AND = 0.08

# tetO regulation parameters (/uM-sec, /sec)
k_rep_T = 2.98
k_derep_T = 0.001
k_ind_T = 4.0
k_unin_T = 0.022

# smtO regulation parameters (/uM-sec, /sec)
k_rep_S = 3.0
k_derep_S = 0.18
k_ind_S = 80.0
k_unin_S = 0.1

# ==========================================
# 2. Species Definition
# ==========================================
# tetO-regulated Species
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetR_tet")
RNAP = Species("RNAP")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# smtO-regulated Species
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# Strand Displacements Species
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# Combine all species into a list for the CRN
species_list = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
]

# ==========================================
# 3. Reactions Definition
# ==========================================
reactions = [
    # --- tetO-regulated Template ---
    # TetR + D_tetO <---> TetRD
    Reaction(inputs=[TetR, D_tetO], outputs=[TetRD], k=k_rep_T, k_rev=k_derep_T),
    
    # TetR + tet <---> TetR-tet
    Reaction(inputs=[TetR, tet], outputs=[TetR_tet], k=k_ind_T, k_rev=k_unin_T),
    
    # RNAP + D_tetO <---> RD_tetO
    Reaction(inputs=[RNAP, D_tetO], outputs=[RD_tetO], k=k_bind, k_rev=k_unbind),
    
    # RD_tetO ---> InvR_T + RNAP + D_tetO
    Reaction(inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO], k=k_m_tetO),

    # --- smtO-Regulated Template ---
    # SmtB + D_smtO <---> SmtBD
    Reaction(inputs=[SmtB, D_smtO], outputs=[SmtBD], k=k_rep_S, k_rev=k_derep_S),
    
    # SmtB + Zn <---> SmtB-Zn
    Reaction(inputs=[SmtB, Zn], outputs=[SmtB_Zn], k=k_ind_S, k_rev=k_unin_S),
    
    # RNAP + D_smtO <---> RD_smtO
    Reaction(inputs=[RNAP, D_smtO], outputs=[RD_smtO], k=k_bind, k_rev=k_unbind),
    
    # RD_smtO ---> InvR_S + RNAP + D_smtO
    Reaction(inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO], k=k_m_smtO),

    # --- Strand Displacements ---
    # InvR_T + InvR_S + AND-Gate ---> InvR_D + Gated_TS
    Reaction(inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS], k=k_SD_AND),
    
    # InvR_D + RQ ---> SD + Q
    Reaction(inputs=[InvR_D, RQ], outputs=[SD, Q], k=k_SD)
]

# ==========================================
# 4. Construct and Print the CRN
# ==========================================
# Initialize the Chemical Reaction Network
crn = ChemicalReactionNetwork(species=species_list, reactions=reactions)

# Print a formatted string of the network (useful for validation)
print(crn.pretty_print(show_rates=True))

crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r3.xml') # export manually added

TypeError: Reaction.__init__() got an unexpected keyword argument 'k'

### Attempt 4 [Successful]

In [5]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Define all Species
# ==========================================

# tetO-regulated Template Species
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetR_tet")  # Hyphen replaced with underscore for valid variable naming
RNAP = Species("RNAP")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# smtO-regulated Template Species
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# Strand Displacement Species
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# Combine all species into a list
species_list = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
]

# ==========================================
# 2. Define Kinetics Parameters
# ==========================================

# tetO Parameters
k_m_tetO = 0.1
k_rep_T = 2.98
k_derep_T = 0.001
k_ind_T = 4.0
k_unin_T = 0.022

# smtO Parameters
k_m_smtO = 0.04
k_rep_S = 3.0
k_derep_S = 0.18
k_ind_S = 80.0
k_unin_S = 0.1

# General binding/unbinding
k_bind = 56.0
k_unbind = 0.2

# Strand Displacement Parameters
k_SD = 0.04
k_SD_AND = 0.08


# ==========================================
# 3. Define Reactions
# ==========================================
reactions_list = []

# --- tetO-regulated Template Reactions ---

# TetR + D_tetO <---> TetRD
reactions_list.append(
    Reaction.from_massaction(inputs=[TetR, D_tetO], outputs=[TetRD], k_forward=k_rep_T, k_reverse=k_derep_T)
)

# TetR + tet <---> TetR-tet
reactions_list.append(
    Reaction.from_massaction(inputs=[TetR, tet], outputs=[TetR_tet], k_forward=k_ind_T, k_reverse=k_unin_T)
)

# RNAP + D_tetO <---> RD_tetO
reactions_list.append(
    Reaction.from_massaction(inputs=[RNAP, D_tetO], outputs=[RD_tetO], k_forward=k_bind, k_reverse=k_unbind)
)

# RD_tetO ---> InvR_T + RNAP + D_tetO
reactions_list.append(
    Reaction.from_massaction(inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO], k_forward=k_m_tetO)
)

# --- smtO-Regulated Template Reactions ---

# SmtB + D_smtO <---> SmtBD
reactions_list.append(
    Reaction.from_massaction(inputs=[SmtB, D_smtO], outputs=[SmtBD], k_forward=k_rep_S, k_reverse=k_derep_S)
)

# SmtB + Zn <---> SmtB-Zn
reactions_list.append(
    Reaction.from_massaction(inputs=[SmtB, Zn], outputs=[SmtB_Zn], k_forward=k_ind_S, k_reverse=k_unin_S)
)

# RNAP + D_smtO <---> RD_smtO
reactions_list.append(
    Reaction.from_massaction(inputs=[RNAP, D_smtO], outputs=[RD_smtO], k_forward=k_bind, k_reverse=k_unbind)
)

# RD_smtO ---> InvR_S + RNAP + D_smtO
reactions_list.append(
    Reaction.from_massaction(inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO], k_forward=k_m_smtO)
)

# --- Strand Displacements ---

# InvR_T + InvR_S + AND-Gate ---> InvR_D + Gated_TS
reactions_list.append(
    Reaction.from_massaction(inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS], k_forward=k_SD_AND)
)

# InvR_D + RQ ---> SD + Q
reactions_list.append(
    Reaction.from_massaction(inputs=[InvR_D, RQ], outputs=[SD, Q], k_forward=k_SD)
)

# ==========================================
# 4. Compile the CRN
# ==========================================

crn = ChemicalReactionNetwork(species=species_list, reactions=reactions_list)

# Print the text representation to verify
print("Chemical Reaction Network compiled successfully!\n")
print(crn.pretty_print())

crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r4.xml') # export manually added

Chemical Reaction Network compiled successfully!

Species(N = 21) = {
tet (@ 0),  Zn (@ 0),  TetR_tet (@ 0),  TetRD (@ 0),  TetR (@ 0),  SmtB_Zn (@ 0),  SmtBD (@ 0),  SmtB (@ 0),  SD (@ 0),  RQ (@ 0),  RNAP (@ 0),  RD_tetO (@ 0),  RD_smtO (@ 0),  Q (@ 0),  InvR_T (@ 0),  InvR_S (@ 0),  InvR_D (@ 0),  Gated_TS (@ 0),  D_tetO (@ 0),  D_smtO (@ 0),  AND_Gate (@ 0),  
}

Reactions (10) = [
0. TetR+D_tetO <--> TetRD
 Kf=k_forward * TetR * D_tetO
 Kr=k_reverse * TetRD
  k_forward=2.98
  k_reverse=0.001

1. TetR+tet <--> TetR_tet
 Kf=k_forward * TetR * tet
 Kr=k_reverse * TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. RNAP+D_tetO <--> RD_tetO
 Kf=k_forward * RNAP * D_tetO
 Kr=k_reverse * RD_tetO
  k_forward=56.0
  k_reverse=0.2

3. RD_tetO --> InvR_T+RNAP+D_tetO
 Kf=k_forward * RD_tetO
  k_forward=0.1

4. SmtB+D_smtO <--> SmtBD
 Kf=k_forward * SmtB * D_smtO
 Kr=k_reverse * SmtBD
  k_forward=3.0
  k_reverse=0.18

5. SmtB+Zn <--> SmtB_Zn
 Kf=k_forward * SmtB * Zn
 Kr=k_reverse * SmtB_Zn
  k_for

True

### Attempt 5 [Successful]

In [6]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Kinetic Parameters
# ==========================================
k_m_tetO = 0.1       # /sec
k_m_smtO = 0.04      # /sec
k_bind = 56.0        # /uM-sec
k_unbind = 0.2       # /sec

k_SD = 0.04          # /uM-sec
k_SD_AND = 0.08      # /uM-sec

# tetO parameters
k_rep_T = 2.98       # /uM-sec
k_derep_T = 0.001    # /uM-sec
k_ind_T = 4.0        # /uM-sec
k_unin_T = 0.022     # /sec

# smtO parameters
k_rep_S = 3.0        # /uM-sec
k_derep_S = 0.18     # /uM-sec
k_ind_S = 80.0       # /uM-sec
k_unin_S = 0.1       # /sec

# ==========================================
# 2. Define Species
# ==========================================
# Shared machinery
RNAP = Species("RNAP")

# tetO-regulated Template species
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetR_tet")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# smtO-regulated Template species
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# Strand Displacement species
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# Combine all species into a list
species_list = [
    RNAP, TetR, D_tetO, TetRD, tet, TetR_tet, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
]

# ==========================================
# 3. Define Reactions
# ==========================================
reactions_list = []

# --- tetO-regulated Template Reactions ---
# TetR + D_tetO <--> TetRD
reactions_list.append(
    Reaction.from_massaction([TetR, D_tetO], [TetRD], k_forward=k_rep_T, k_reverse=k_derep_T)
)
# TetR + tet <--> TetR-tet
reactions_list.append(
    Reaction.from_massaction([TetR, tet], [TetR_tet], k_forward=k_ind_T, k_reverse=k_unin_T)
)
# RNAP + D_tetO <--> RD_tetO
reactions_list.append(
    Reaction.from_massaction([RNAP, D_tetO], [RD_tetO], k_forward=k_bind, k_reverse=k_unbind)
)
# RD_tetO --> InvR_T + RNAP + D_tetO
reactions_list.append(
    Reaction.from_massaction([RD_tetO], [InvR_T, RNAP, D_tetO], k_forward=k_m_tetO)
)

# --- smtO-regulated Template Reactions ---
# SmtB + D_smtO <--> SmtBD
reactions_list.append(
    Reaction.from_massaction([SmtB, D_smtO], [SmtBD], k_forward=k_rep_S, k_reverse=k_derep_S)
)
# SmtB + Zn <--> SmtB-Zn
reactions_list.append(
    Reaction.from_massaction([SmtB, Zn], [SmtB_Zn], k_forward=k_ind_S, k_reverse=k_unin_S)
)
# RNAP + D_smtO <--> RD_smtO
reactions_list.append(
    Reaction.from_massaction([RNAP, D_smtO], [RD_smtO], k_forward=k_bind, k_reverse=k_unbind)
)
# RD_smtO --> InvR_S + RNAP + D_smtO
reactions_list.append(
    Reaction.from_massaction([RD_smtO], [InvR_S, RNAP, D_smtO], k_forward=k_m_smtO)
)

# --- Strand Displacement Reactions ---
# InvR_T + InvR_S + AND-Gate --> InvR_D + Gated_TS
reactions_list.append(
    Reaction.from_massaction([InvR_T, InvR_S, AND_Gate], [InvR_D, Gated_TS], k_forward=k_SD_AND)
)
# InvR_D + RQ --> SD + Q
reactions_list.append(
    Reaction.from_massaction([InvR_D, RQ], [SD, Q], k_forward=k_SD)
)

# ==========================================
# 4. Compile Chemical Reaction Network
# ==========================================
crn = ChemicalReactionNetwork(species=species_list, reactions=reactions_list)

# Print out a string representation to verify the CRN
print("Successfully built the CRN:")
print(crn)

crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r5.xml') # export manually added

Successfully built the CRN:
Species = RNAP, TetR, D_tetO, TetRD, tet, TetR_tet, RD_tetO, InvR_T, SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S, AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
Reactions = [
	TetR+D_tetO <--> TetRD
	TetR+tet <--> TetR_tet
	RNAP+D_tetO <--> RD_tetO
	RD_tetO --> InvR_T+RNAP+D_tetO
	SmtB+D_smtO <--> SmtBD
	SmtB+Zn <--> SmtB_Zn
	RNAP+D_smtO <--> RD_smtO
	RD_smtO --> InvR_S+RNAP+D_smtO
	InvR_T+InvR_S+AND_Gate --> InvR_D+Gated_TS
	InvR_D+RQ --> SD+Q
]


True

### Attempt 6 [Failed]

In [7]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Define Kinetic Parameters
# ==========================================
# tetO-regulated Template
k_m_tetO = 0.1       # /sec
k_rep_T = 2.98       # /uM-sec
k_derep_T = 0.001    # /uM-sec
k_ind_T = 4.0        # /uM-sec
k_unin_T = 0.022     # /sec

# smtO-Regulated Template
k_m_smtO = 0.04      # /sec
k_rep_S = 3.0        # /uM-sec
k_derep_S = 0.18     # /uM-sec
k_ind_S = 80.0       # /uM-sec
k_unin_S = 0.1       # /sec

# Shared / General Parameters
k_bind = 56.0        # /uM-sec
k_unbind = 0.2       # /sec

# Strand Displacements
k_SD = 0.04          # /uM-sec
k_SD_AND = 0.08      # /uM-sec


# ==========================================
# 2. Define Chemical Species
# ==========================================
# RNAP
RNAP = Species("RNAP")

# tetO-regulated Species
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetR_tet")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# smtO-Regulated Species
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# Strand Displacement Species
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")


# ==========================================
# 3. Define Reactions
# ==========================================
reactions = []

# --- tetO-regulated Template Reactions ---
# TetR + D_tetO <--> TetRD
reactions.append(Reaction(inputs=[TetR, D_tetO], outputs=[TetRD], k=k_rep_T, k_rev=k_derep_T))

# TetR + tet <--> TetR-tet
reactions.append(Reaction(inputs=[TetR, tet], outputs=[TetR_tet], k=k_ind_T, k_rev=k_unin_T))

# RNAP + D_tetO <--> RD_tetO
reactions.append(Reaction(inputs=[RNAP, D_tetO], outputs=[RD_tetO], k=k_bind, k_rev=k_unbind))

# RD_tetO --> InvR_T + RNAP + D_tetO
reactions.append(Reaction(inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO], k=k_m_tetO))


# --- smtO-Regulated Template Reactions ---
# SmtB + D_smtO <--> SmtBD
reactions.append(Reaction(inputs=[SmtB, D_smtO], outputs=[SmtBD], k=k_rep_S, k_rev=k_derep_S))

# SmtB + Zn <--> SmtB-Zn
reactions.append(Reaction(inputs=[SmtB, Zn], outputs=[SmtB_Zn], k=k_ind_S, k_rev=k_unin_S))

# RNAP + D_smtO <--> RD_smtO
reactions.append(Reaction(inputs=[RNAP, D_smtO], outputs=[RD_smtO], k=k_bind, k_rev=k_unbind))

# RD_smtO --> InvR_S + RNAP + D_smtO
reactions.append(Reaction(inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO], k=k_m_smtO))


# --- Strand Displacements ---
# InvR_T + InvR_S + AND-Gate --> InvR_D + Gated_TS
reactions.append(Reaction(inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS], k=k_SD_AND))

# InvR_D + RQ --> SD + Q
reactions.append(Reaction(inputs=[InvR_D, RQ], outputs=[SD, Q], k=k_SD))


# ==========================================
# 4. Compile the Chemical Reaction Network
# ==========================================
# Collect all distinct species dynamically from the reactions
species_list = list(set([s for r in reactions for s in r.inputs + r.outputs]))

# Build the CRN
crn = ChemicalReactionNetwork(species=species_list, reactions=reactions)

# Print a text representation to verify
print("Chemical Reaction Network compiled successfully!")
print(crn)

crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r6.xml') # export manually added

TypeError: Reaction.__init__() got an unexpected keyword argument 'k'

### Attempt 7 [Successful]

In [8]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Define Kinetic Parameters
# ==========================================
# Transcription & Binding Parameters
km_tetO = 0.1       # /sec
km_smtO = 0.04      # /sec
k_bind = 56.0       # /uM-sec
k_unbind = 0.2      # /sec

# Strand Displacement Parameters
k_SD = 0.04         # /uM-sec
k_SD_AND = 0.08     # /uM^2-sec (Assuming standard 3-body mass action)

# tetO Regulatory Parameters
k_rep_T = 2.98      # /uM-sec
k_derep_T = 0.001   # /sec
k_ind_T = 4.0       # /uM-sec
k_unin_T = 0.022    # /sec

# smtO Regulatory Parameters
k_rep_S = 3.0       # /uM-sec
k_derep_S = 0.18    # /sec
k_ind_S = 80.0      # /uM-sec
k_unin_S = 0.1      # /sec

# ==========================================
# 2. Define Chemical Species
# ==========================================
# Shared machinery
RNAP = Species("RNAP")

# tetO-regulated Template Species
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetR_tet")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# smtO-Regulated Template Species
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# Strand Displacement Species
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# Combine all species into a list
all_species = [
    RNAP, TetR, D_tetO, TetRD, tet, TetR_tet, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
]

# ==========================================
# 3. Define Reactions
# ==========================================
rxns = []

# --- tetO-regulated Template Reactions ---
# TetR + D_tetO <--> TetRD
rxns.append(Reaction.from_massaction([TetR, D_tetO], [TetRD], k_forward=k_rep_T, k_reverse=k_derep_T))
# TetR + tet <--> TetR-tet
rxns.append(Reaction.from_massaction([TetR, tet], [TetR_tet], k_forward=k_ind_T, k_reverse=k_unin_T))
# RNAP + D_tetO <--> RD_tetO
rxns.append(Reaction.from_massaction([RNAP, D_tetO], [RD_tetO], k_forward=k_bind, k_reverse=k_unbind))
# RD_tetO --> InvR_T + RNAP + D_tetO
rxns.append(Reaction.from_massaction([RD_tetO], [InvR_T, RNAP, D_tetO], k_forward=km_tetO))

# --- smtO-Regulated Template Reactions ---
# SmtB + D_smtO <--> SmtBD
rxns.append(Reaction.from_massaction([SmtB, D_smtO], [SmtBD], k_forward=k_rep_S, k_reverse=k_derep_S))
# SmtB + Zn <--> SmtB-Zn
rxns.append(Reaction.from_massaction([SmtB, Zn], [SmtB_Zn], k_forward=k_ind_S, k_reverse=k_unin_S))
# RNAP + D_smtO <--> RD_smtO
rxns.append(Reaction.from_massaction([RNAP, D_smtO], [RD_smtO], k_forward=k_bind, k_reverse=k_unbind))
# RD_smtO --> InvR_S + RNAP + D_smtO
rxns.append(Reaction.from_massaction([RD_smtO], [InvR_S, RNAP, D_smtO], k_forward=km_smtO))

# --- Strand Displacement Reactions ---
# InvR_T + InvR_S + AND-Gate --> InvR_D + Gated_TS (3-body reaction)
rxns.append(Reaction.from_massaction([InvR_T, InvR_S, AND_Gate], [InvR_D, Gated_TS], k_forward=k_SD_AND))
# InvR_D + RQ --> SD + Q
rxns.append(Reaction.from_massaction([InvR_D, RQ], [SD, Q], k_forward=k_SD))

# ==========================================
# 4. Compile the CRN
# ==========================================
crn = ChemicalReactionNetwork(species=all_species, reactions=rxns)

# Print the network to verify
print("Compiled CRN:")
print(crn.pretty_print())

crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r7.xml') # export manually added

Compiled CRN:
Species(N = 21) = {
tet (@ 0),  Zn (@ 0),  TetR_tet (@ 0),  TetRD (@ 0),  TetR (@ 0),  SmtB_Zn (@ 0),  SmtBD (@ 0),  SmtB (@ 0),  SD (@ 0),  RQ (@ 0),  RNAP (@ 0),  RD_tetO (@ 0),  RD_smtO (@ 0),  Q (@ 0),  InvR_T (@ 0),  InvR_S (@ 0),  InvR_D (@ 0),  Gated_TS (@ 0),  D_tetO (@ 0),  D_smtO (@ 0),  AND_Gate (@ 0),  
}

Reactions (10) = [
0. TetR+D_tetO <--> TetRD
 Kf=k_forward * TetR * D_tetO
 Kr=k_reverse * TetRD
  k_forward=2.98
  k_reverse=0.001

1. TetR+tet <--> TetR_tet
 Kf=k_forward * TetR * tet
 Kr=k_reverse * TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. RNAP+D_tetO <--> RD_tetO
 Kf=k_forward * RNAP * D_tetO
 Kr=k_reverse * RD_tetO
  k_forward=56.0
  k_reverse=0.2

3. RD_tetO --> InvR_T+RNAP+D_tetO
 Kf=k_forward * RD_tetO
  k_forward=0.1

4. SmtB+D_smtO <--> SmtBD
 Kf=k_forward * SmtB * D_smtO
 Kr=k_reverse * SmtBD
  k_forward=3.0
  k_reverse=0.18

5. SmtB+Zn <--> SmtB_Zn
 Kf=k_forward * SmtB * Zn
 Kr=k_reverse * SmtB_Zn
  k_forward=80.0
  k_reverse=0.1

6. RNAP+D

True

### Attempt 8 [Successful]

In [9]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Define Kinetic Parameters
# ==========================================
k_m_tetO = 0.1        # /sec
k_m_smtO = 0.04       # /sec
k_bind = 56.0         # /uM-sec
k_unbind = 0.2        # /sec

k_SD = 0.04           # /uM-sec
k_SD_AND = 0.08       # /uM-sec

k_rep_T = 2.98        # /uM-sec
k_derep_T = 0.001     # /uM-sec
k_ind_T = 4.0         # /uM-sec
k_unin_T = 0.022      # /sec

k_rep_S = 3.0         # /uM-sec
k_derep_S = 0.18      # /uM-sec
k_ind_S = 80.0        # /uM-sec
k_unin_S = 0.1        # /sec

# ==========================================
# 2. Define Species
# ==========================================
# tetO-regulated Template Species
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetR_tet")
RNAP = Species("RNAP")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# smtO-regulated Template Species
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# Strand Displacement Species
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# Aggregate all species into a single list
species_list = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
]

# ==========================================
# 3. Define Reactions
# ==========================================
rxns = [
    # -- tetO-regulated Template --
    # TetR + D_tetO <---> TetRD
    Reaction.from_massaction(inputs=[TetR, D_tetO], outputs=[TetRD], 
                             k_forward=k_rep_T, k_reverse=k_derep_T),
    
    # TetR + tet <---> TetR-tet
    Reaction.from_massaction(inputs=[TetR, tet], outputs=[TetR_tet], 
                             k_forward=k_ind_T, k_reverse=k_unin_T),
    
    # RNAP + D_tetO <---> RD_tetO
    Reaction.from_massaction(inputs=[RNAP, D_tetO], outputs=[RD_tetO], 
                             k_forward=k_bind, k_reverse=k_unbind),
    
    # RD_tetO ---> InvR_T + RNAP + D_tetO
    Reaction.from_massaction(inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO], 
                             k_forward=k_m_tetO),

    # -- smtO-Regulated Template --
    # SmtB + D_smtO <---> SmtBD
    Reaction.from_massaction(inputs=[SmtB, D_smtO], outputs=[SmtBD], 
                             k_forward=k_rep_S, k_reverse=k_derep_S),
    
    # SmtB + Zn <---> SmtB-Zn
    Reaction.from_massaction(inputs=[SmtB, Zn], outputs=[SmtB_Zn], 
                             k_forward=k_ind_S, k_reverse=k_unin_S),
    
    # RNAP + D_smtO <---> RD_smtO
    Reaction.from_massaction(inputs=[RNAP, D_smtO], outputs=[RD_smtO], 
                             k_forward=k_bind, k_reverse=k_unbind),
    
    # RD_smtO ---> InvR_S + RNAP + D_smtO
    Reaction.from_massaction(inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO], 
                             k_forward=k_m_smtO),

    # -- Strand Displacements --
    # InvR_T + InvR_S + AND-Gate ---> InvR_D + Gated_TS
    Reaction.from_massaction(inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS], 
                             k_forward=k_SD_AND),
    
    # InvR_D + RQ ---> SD + Q
    Reaction.from_massaction(inputs=[InvR_D, RQ], outputs=[SD, Q], 
                             k_forward=k_SD)
]

# ==========================================
# 4. Compile the Chemical Reaction Network
# ==========================================
crn = ChemicalReactionNetwork(species=species_list, reactions=rxns)

# Print a summary of the network (useful for verifying construction)
print(crn.pretty_print())

crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r8.xml') # export manually added

Species(N = 21) = {
tet (@ 0),  Zn (@ 0),  TetR_tet (@ 0),  TetRD (@ 0),  TetR (@ 0),  SmtB_Zn (@ 0),  SmtBD (@ 0),  SmtB (@ 0),  SD (@ 0),  RQ (@ 0),  RNAP (@ 0),  RD_tetO (@ 0),  RD_smtO (@ 0),  Q (@ 0),  InvR_T (@ 0),  InvR_S (@ 0),  InvR_D (@ 0),  Gated_TS (@ 0),  D_tetO (@ 0),  D_smtO (@ 0),  AND_Gate (@ 0),  
}

Reactions (10) = [
0. TetR+D_tetO <--> TetRD
 Kf=k_forward * TetR * D_tetO
 Kr=k_reverse * TetRD
  k_forward=2.98
  k_reverse=0.001

1. TetR+tet <--> TetR_tet
 Kf=k_forward * TetR * tet
 Kr=k_reverse * TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. RNAP+D_tetO <--> RD_tetO
 Kf=k_forward * RNAP * D_tetO
 Kr=k_reverse * RD_tetO
  k_forward=56.0
  k_reverse=0.2

3. RD_tetO --> InvR_T+RNAP+D_tetO
 Kf=k_forward * RD_tetO
  k_forward=0.1

4. SmtB+D_smtO <--> SmtBD
 Kf=k_forward * SmtB * D_smtO
 Kr=k_reverse * SmtBD
  k_forward=3.0
  k_reverse=0.18

5. SmtB+Zn <--> SmtB_Zn
 Kf=k_forward * SmtB * Zn
 Kr=k_reverse * SmtB_Zn
  k_forward=80.0
  k_reverse=0.1

6. RNAP+D_smtO <--> RD_

True

### Attempt 9 [Failed]

In [10]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Define Chemical Species
# ==========================================

# tetO-regulated Template Species
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetR_tet")
RNAP = Species("RNAP")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# smtO-Regulated Template Species
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# Strand Displacement Species
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# Group all species into a single list
all_species = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
]


# ==========================================
# 2. Define Kinetic Parameters
# ==========================================

k_m_tetO = 0.1       # /sec
k_m_smtO = 0.04      # /sec
k_bind = 56.0        # /uM-sec
k_unbind = 0.2       # /sec

k_SD = 0.04          # /uM-sec
k_SD_AND = 0.08      # /uM-sec

k_rep_T = 2.98       # /uM-sec
k_derep_T = 0.001    # /uM-sec
k_ind_T = 4.0        # /uM-sec
k_unin_T = 0.022     # /sec

k_rep_S = 3.0        # /uM-sec
k_derep_S = 0.18     # /uM-sec
k_ind_S = 80.0       # /uM-sec
k_unin_S = 0.1       # /sec


# ==========================================
# 3. Construct Reactions
# ==========================================
# Note: Reversible reactions are split into two unidirectional 
# reactions for maximal compatibility across BioCRNpyler versions.

reactions = [
    # ------------------------------------------
    # tetO-regulated Template
    # ------------------------------------------
    # TetR + D_tetO <---> TetRD
    Reaction([TetR, D_tetO], [TetRD], k=k_rep_T),
    Reaction([TetRD], [TetR, D_tetO], k=k_derep_T),
    
    # TetR + tet <---> TetR-tet
    Reaction([TetR, tet], [TetR_tet], k=k_ind_T),
    Reaction([TetR_tet], [TetR, tet], k=k_unin_T),
    
    # RNAP + D_tetO <---> RD_tetO
    Reaction([RNAP, D_tetO], [RD_tetO], k=k_bind),
    Reaction([RD_tetO], [RNAP, D_tetO], k=k_unbind),
    
    # RD_tetO ---> InvR_T + RNAP + D_tetO
    Reaction([RD_tetO], [InvR_T, RNAP, D_tetO], k=k_m_tetO),

    # ------------------------------------------
    # smtO-Regulated Template
    # ------------------------------------------
    # SmtB + D_smtO <---> SmtBD
    Reaction([SmtB, D_smtO], [SmtBD], k=k_rep_S),
    Reaction([SmtBD], [SmtB, D_smtO], k=k_derep_S),
    
    # SmtB + Zn <---> SmtB-Zn
    Reaction([SmtB, Zn], [SmtB_Zn], k=k_ind_S),
    Reaction([SmtB_Zn], [SmtB, Zn], k=k_unin_S),
    
    # RNAP + D_smtO <---> RD_smtO
    Reaction([RNAP, D_smtO], [RD_smtO], k=k_bind),
    Reaction([RD_smtO], [RNAP, D_smtO], k=k_unbind),
    
    # RD_smtO ---> InvR_S + RNAP + D_smtO
    Reaction([RD_smtO], [InvR_S, RNAP, D_smtO], k=k_m_smtO),

    # ------------------------------------------
    # Strand Displacements
    # ------------------------------------------
    # InvR_T + InvR_S + AND-Gate ---> InvR_D + Gated_TS
    Reaction([InvR_T, InvR_S, AND_Gate], [InvR_D, Gated_TS], k=k_SD_AND),
    
    # InvR_D + RQ ---> SD + Q
    Reaction([InvR_D, RQ], [SD, Q], k=k_SD)
]


# ==========================================
# 4. Build and Compile the CRN
# ==========================================

crn = ChemicalReactionNetwork(species=all_species, reactions=reactions)

# Print a formatted summary of the network structure
if __name__ == "__main__":
    print(crn.pretty_print())

crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r9.xml') # export manually added

TypeError: Reaction.__init__() got an unexpected keyword argument 'k'

### Attempt 10 [Successful]

In [11]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ==========================================
# 1. Define Kinetic Parameters
# ==========================================

# Transcription & Binding Parameters
k_m_tetO = 0.1        # /sec
k_m_smtO = 0.04       # /sec
k_bind = 56           # /uM-sec
k_unbind = 0.2        # /sec

# Strand Displacement Parameters
k_SD = 0.04           # /uM-sec
k_SD_AND = 0.08       # /uM-sec

# tetO-Regulated Parameters
k_rep_T = 2.98        # /uM-sec
k_derep_T = 0.001     # /uM-sec
k_ind_T = 4.0         # /uM-sec
k_unin_T = 0.022      # /sec

# smtO-Regulated Parameters
k_rep_S = 3.0         # /uM-sec
k_derep_S = 0.18      # /uM-sec
k_ind_S = 80          # /uM-sec
k_unin_S = 0.1        # /sec


# ==========================================
# 2. Define Species
# ==========================================

# Core Machinery
RNAP = Species("RNAP")

# tetO System Species
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetR_tet")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# smtO System Species
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# Strand Displacement / Logic Gate Species
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

species_list = [
    RNAP, TetR, D_tetO, TetRD, tet, TetR_tet, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
]


# ==========================================
# 3. Define Reactions
# ==========================================
reactions = []

# --- tetO-regulated Template Reactions ---

# TetR + D_tetO <---> TetRD
rxn_tetO_rep = Reaction.from_massaction(
    inputs=[TetR, D_tetO], outputs=[TetRD], 
    k_forward=k_rep_T, k_reverse=k_derep_T
)
reactions.append(rxn_tetO_rep)

# TetR + tet <---> TetR-tet
rxn_tetO_ind = Reaction.from_massaction(
    inputs=[TetR, tet], outputs=[TetR_tet], 
    k_forward=k_ind_T, k_reverse=k_unin_T
)
reactions.append(rxn_tetO_ind)

# RNAP + D_tetO <---> RD_tetO
rxn_tetO_bind = Reaction.from_massaction(
    inputs=[RNAP, D_tetO], outputs=[RD_tetO], 
    k_forward=k_bind, k_reverse=k_unbind
)
reactions.append(rxn_tetO_bind)

# RD_tetO ---> InvR_T + RNAP + D_tetO
rxn_tetO_tx = Reaction.from_massaction(
    inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO], 
    k_forward=k_m_tetO
)
reactions.append(rxn_tetO_tx)


# --- smtO-Regulated Template Reactions ---

# SmtB + D_smtO <---> SmtBD
rxn_smtO_rep = Reaction.from_massaction(
    inputs=[SmtB, D_smtO], outputs=[SmtBD], 
    k_forward=k_rep_S, k_reverse=k_derep_S
)
reactions.append(rxn_smtO_rep)

# SmtB + Zn <---> SmtB-Zn
rxn_smtO_ind = Reaction.from_massaction(
    inputs=[SmtB, Zn], outputs=[SmtB_Zn], 
    k_forward=k_ind_S, k_reverse=k_unin_S
)
reactions.append(rxn_smtO_ind)

# RNAP + D_smtO <---> RD_smtO
rxn_smtO_bind = Reaction.from_massaction(
    inputs=[RNAP, D_smtO], outputs=[RD_smtO], 
    k_forward=k_bind, k_reverse=k_unbind
)
reactions.append(rxn_smtO_bind)

# RD_smtO ---> InvR_S + RNAP + D_smtO
rxn_smtO_tx = Reaction.from_massaction(
    inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO], 
    k_forward=k_m_smtO
)
reactions.append(rxn_smtO_tx)


# --- Strand Displacements ---

# InvR_T + InvR_S + AND-Gate ---> InvR_D + Gated_TS
rxn_sd_and = Reaction.from_massaction(
    inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS], 
    k_forward=k_SD_AND
)
reactions.append(rxn_sd_and)

# InvR_D + RQ ---> SD + Q
rxn_sd_rep = Reaction.from_massaction(
    inputs=[InvR_D, RQ], outputs=[SD, Q], 
    k_forward=k_SD
)
reactions.append(rxn_sd_rep)


# ==========================================
# 4. Compile the Chemical Reaction Network
# ==========================================

crn = ChemicalReactionNetwork(species=species_list, reactions=reactions)

# Print the text representation of the network to verify
print("Chemical Reaction Network compiled successfully!\n")
print(crn)

crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_gemini_r10.xml') # export manually added

Chemical Reaction Network compiled successfully!

Species = RNAP, TetR, D_tetO, TetRD, tet, TetR_tet, RD_tetO, InvR_T, SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S, AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
Reactions = [
	TetR+D_tetO <--> TetRD
	TetR+tet <--> TetR_tet
	RNAP+D_tetO <--> RD_tetO
	RD_tetO --> InvR_T+RNAP+D_tetO
	SmtB+D_smtO <--> SmtBD
	SmtB+Zn <--> SmtB_Zn
	RNAP+D_smtO <--> RD_smtO
	RD_smtO --> InvR_S+RNAP+D_smtO
	InvR_T+InvR_S+AND_Gate --> InvR_D+Gated_TS
	InvR_D+RQ --> SD+Q
]


True

## ChaptGPT [4/10 Successful]

### Attempt 1 [Successful]

In [12]:
"""
BioCRNpyler implementation of the specified CRN.

This script directly encodes the reactions using ChemicalComplexes and
MassAction kinetics rather than higher-level Components, since the
network consists of explicit biochemical reactions.

Requires:
    pip install biocrnpyler
"""

from biocrnpyler import *

###############################################################################
# Species
###############################################################################

# Proteins / regulators
TetR = Species("TetR", material_type="protein")
SmtB = Species("SmtB", material_type="protein")
RNAP = Species("RNAP", material_type="protein")

# Small molecules
tet = Species("tet")
Zn = Species("Zn")

# DNA templates
D_tetO = Species("D_tetO", material_type="dna")
D_smtO = Species("D_smtO", material_type="dna")

# RNA products
InvR_T = Species("InvR_T", material_type="rna")
InvR_S = Species("InvR_S", material_type="rna")
InvR_D = Species("InvR_D", material_type="rna")

# Strand displacement species
AND_Gate = Species("AND_Gate")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

###############################################################################
# Complexes
###############################################################################

TetRD = Complex([TetR, D_tetO])
TetR_tet = Complex([TetR, tet])
RD_tetO = Complex([RNAP, D_tetO])

SmtBD = Complex([SmtB, D_smtO])
SmtB_Zn = Complex([SmtB, Zn])
RD_smtO = Complex([RNAP, D_smtO])

###############################################################################
# Parameters
###############################################################################

parameters = ParameterDatabase(parameter_dictionary={

    # RNAP binding
    "k_bind": 56.0,
    "k_unbind": 0.2,

    # Transcription
    "k_m_tetO": 0.10,
    "k_m_smtO": 0.04,

    # Strand displacement
    "k_SD": 0.04,
    "k_SD_AND": 0.08,

    # TetR repression / induction
    "k_rep_T": 2.98,
    "k_derep_T": 0.001,
    "k_ind_T": 4.0,
    "k_unin_T": 0.022,

    # SmtB repression / induction
    "k_rep_S": 3.0,
    "k_derep_S": 0.18,
    "k_ind_S": 80.0,
    "k_unin_S": 0.10,
})

###############################################################################
# Reactions
###############################################################################

reactions = []

# ---------------------------------------------------------------------------
# tetO-regulated template
# ---------------------------------------------------------------------------

# TetR + D_tetO <--> TetRD
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=parameters["k_rep_T"].value,
        k_reverse=parameters["k_derep_T"].value,
    )
)

# TetR + tet <--> TetR-tet
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=parameters["k_ind_T"].value,
        k_reverse=parameters["k_unin_T"].value,
    )
)

# RNAP + D_tetO <--> RD_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=parameters["k_bind"].value,
        k_reverse=parameters["k_unbind"].value,
    )
)

# RD_tetO --> InvR_T + RNAP + D_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=parameters["k_m_tetO"].value,
    )
)

# ---------------------------------------------------------------------------
# smtO-regulated template
# ---------------------------------------------------------------------------

# SmtB + D_smtO <--> SmtBD
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=parameters["k_rep_S"].value,
        k_reverse=parameters["k_derep_S"].value,
    )
)

# SmtB + Zn <--> SmtB-Zn
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=parameters["k_ind_S"].value,
        k_reverse=parameters["k_unin_S"].value,
    )
)

# RNAP + D_smtO <--> RD_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=parameters["k_bind"].value,
        k_reverse=parameters["k_unbind"].value,
    )
)

# RD_smtO --> InvR_S + RNAP + D_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=parameters["k_m_smtO"].value,
    )
)

# ---------------------------------------------------------------------------
# Strand displacement
# ---------------------------------------------------------------------------

# InvR_T + InvR_S + AND-Gate --> InvR_D + Gated_TS
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=parameters["k_SD_AND"].value,
    )
)

# InvR_D + RQ --> SD + Q
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=parameters["k_SD"].value,
    )
)

###############################################################################
# Build CRN
###############################################################################

crn = ChemicalReactionNetwork(
    species=[
        TetR,
        SmtB,
        RNAP,
        tet,
        Zn,
        D_tetO,
        D_smtO,
        InvR_T,
        InvR_S,
        InvR_D,
        AND_Gate,
        Gated_TS,
        RQ,
        SD,
        Q,
        TetRD,
        TetR_tet,
        RD_tetO,
        SmtBD,
        SmtB_Zn,
        RD_smtO,
    ],
    reactions=reactions,
)

###############################################################################
# Output
###############################################################################

print("Number of species:", len(crn.species))
print("Number of reactions:", len(crn.reactions))

# Export SBML
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r1.xml')

# Export human-readable CRN
print("\n=== CRN ===")
for r in crn.reactions:
    print(r)

Number of species: 21
Number of reactions: 10

=== CRN ===
protein[TetR]+dna[D_tetO] <--> complex[dna[D_tetO]:protein[TetR]]
protein[TetR]+tet <--> complex[protein[TetR]:tet]
protein[RNAP]+dna[D_tetO] <--> complex[dna[D_tetO]:protein[RNAP]]
complex[dna[D_tetO]:protein[RNAP]] --> rna[InvR_T]+protein[RNAP]+dna[D_tetO]
protein[SmtB]+dna[D_smtO] <--> complex[dna[D_smtO]:protein[SmtB]]
protein[SmtB]+Zn <--> complex[Zn:protein[SmtB]]
protein[RNAP]+dna[D_smtO] <--> complex[dna[D_smtO]:protein[RNAP]]
complex[dna[D_smtO]:protein[RNAP]] --> rna[InvR_S]+protein[RNAP]+dna[D_smtO]
rna[InvR_T]+rna[InvR_S]+AND_Gate --> rna[InvR_D]+Gated_TS
rna[InvR_D]+RQ --> SD+Q


### Attempt 2 [Failed]

In [13]:
"""
BioCRNpyler implementation of the specified CRN.

This script explicitly constructs all species and reactions using
BioCRNpyler's Species, Reaction, and ChemicalReactionNetwork classes.

Reaction scheme implemented:

tetO template:
TetR + D_tetO <-> TetRD
TetR + tet <-> TetR_tet
RNAP + D_tetO <-> RD_tetO
RD_tetO -> InvR_T + RNAP + D_tetO

smtO template:
SmtB + D_smtO <-> SmtBD
SmtB + Zn <-> SmtB_Zn
RNAP + D_smtO <-> RD_smtO
RD_smtO -> InvR_S + RNAP + D_smtO

Strand displacement:
InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS
InvR_D + RQ -> SD + Q
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

###############################################################################
# Species
###############################################################################

TetR = Species("TetR")
tet = Species("tet")
TetR_tet = Species("TetR_tet")

SmtB = Species("SmtB")
Zn = Species("Zn")
SmtB_Zn = Species("SmtB_Zn")

RNAP = Species("RNAP")

D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")

RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

###############################################################################
# Parameters
###############################################################################

k_m_tetO = 0.10

k_m_smtO = 0.04

k_bind = 56.0
k_unbind = 0.2

k_SD = 0.04
k_SD_AND = 0.08

k_rep_T = 2.98
k_derep_T = 0.001

k_ind_T = 4.0
k_unin_T = 0.022

k_rep_S = 3.0
k_derep_S = 0.18

k_ind_S = 80.0
k_unin_S = 0.10

###############################################################################
# Reactions
###############################################################################

reactions = [

    ###########################################################################
    # tetO-regulated template
    ###########################################################################

    Reaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=k_rep_T,
        k_reverse=k_derep_T
    ),

    Reaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=k_ind_T,
        k_reverse=k_unin_T
    ),

    Reaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=k_bind,
        k_reverse=k_unbind
    ),

    Reaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO
    ),

    ###########################################################################
    # smtO-regulated template
    ###########################################################################

    Reaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=k_rep_S,
        k_reverse=k_derep_S
    ),

    Reaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=k_ind_S,
        k_reverse=k_unin_S
    ),

    Reaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=k_bind,
        k_reverse=k_unbind
    ),

    Reaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO
    ),

    ###########################################################################
    # Strand displacement
    ###########################################################################

    Reaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND
    ),

    Reaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=k_SD
    ),
]

###############################################################################
# Construct CRN
###############################################################################

species = [
    TetR,
    tet,
    TetR_tet,
    SmtB,
    Zn,
    SmtB_Zn,
    RNAP,
    D_tetO,
    TetRD,
    RD_tetO,
    InvR_T,
    D_smtO,
    SmtBD,
    RD_smtO,
    InvR_S,
    AND_Gate,
    InvR_D,
    Gated_TS,
    RQ,
    SD,
    Q,
]

crn = ChemicalReactionNetwork(
    species=species,
    reactions=reactions
)

###############################################################################
# Display CRN
###############################################################################

print(crn)

# Optional SBML export (requires libSBML)
# from biocrnpyler import CRNLab
# Export SBML
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r2.xml')

TypeError: Reaction.__init__() got an unexpected keyword argument 'k_forward'

### Attempt 3 [Successful]

In [14]:
"""
BioCRNpyler implementation of the CRN described in the prompt.

This script builds the CRN directly from the specified reactions using
ChemicalReaction objects (rather than higher-level Components).

Compatible with BioCRNpyler 1.x.
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# =============================================================================
# Species
# =============================================================================

TetR      = Species("TetR")
D_tetO    = Species("D_tetO")
TetRD     = Species("TetRD")
tet       = Species("tet")
TetR_tet  = Species("TetR_tet")
RNAP      = Species("RNAP")
RD_tetO   = Species("RD_tetO")
InvR_T    = Species("InvR_T")

SmtB      = Species("SmtB")
D_smtO    = Species("D_smtO")
SmtBD     = Species("SmtBD")
Zn        = Species("Zn")
SmtB_Zn   = Species("SmtB_Zn")
RD_smtO   = Species("RD_smtO")
InvR_S    = Species("InvR_S")

AND_Gate  = Species("AND_Gate")
InvR_D    = Species("InvR_D")
Gated_TS  = Species("Gated_TS")
RQ        = Species("RQ")
SD        = Species("SD")
Q         = Species("Q")

# =============================================================================
# Parameters
# =============================================================================

parameters = {
    "k_m_tetO":     0.10,
    "k_m_smtO":     0.04,

    "k_bind":       56.0,
    "k_unbind":     0.20,

    "k_SD":         0.04,
    "k_SD_AND":     0.08,

    "k_rep_T":      2.98,
    "k_derep_T":    0.001,

    "k_ind_T":      4.0,
    "k_unin_T":     0.022,

    "k_rep_S":      3.0,
    "k_derep_S":    0.18,

    "k_ind_S":      80.0,
    "k_unin_S":     0.10,
}

# =============================================================================
# Reactions
# =============================================================================

reactions = [

    # -------------------------------------------------------------------------
    # tetO-regulated template
    # -------------------------------------------------------------------------

    # TetR + D_tetO <-> TetRD
    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=parameters["k_rep_T"],
        k_reverse=parameters["k_derep_T"],
    ),

    # TetR + tet <-> TetR_tet
    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=parameters["k_ind_T"],
        k_reverse=parameters["k_unin_T"],
    ),

    # RNAP + D_tetO <-> RD_tetO
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=parameters["k_bind"],
        k_reverse=parameters["k_unbind"],
    ),

    # RD_tetO -> InvR_T + RNAP + D_tetO
    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=parameters["k_m_tetO"],
    ),

    # -------------------------------------------------------------------------
    # smtO-regulated template
    # -------------------------------------------------------------------------

    # SmtB + D_smtO <-> SmtBD
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=parameters["k_rep_S"],
        k_reverse=parameters["k_derep_S"],
    ),

    # SmtB + Zn <-> SmtB_Zn
    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=parameters["k_ind_S"],
        k_reverse=parameters["k_unin_S"],
    ),

    # RNAP + D_smtO <-> RD_smtO
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=parameters["k_bind"],
        k_reverse=parameters["k_unbind"],
    ),

    # RD_smtO -> InvR_S + RNAP + D_smtO
    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=parameters["k_m_smtO"],
    ),

    # -------------------------------------------------------------------------
    # Strand displacement reactions
    # -------------------------------------------------------------------------

    # InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=parameters["k_SD_AND"],
    ),

    # InvR_D + RQ -> SD + Q
    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=parameters["k_SD"],
    ),
]

# =============================================================================
# Collect all species
# =============================================================================

species = [
    TetR,
    D_tetO,
    TetRD,
    tet,
    TetR_tet,
    RNAP,
    RD_tetO,
    InvR_T,
    SmtB,
    D_smtO,
    SmtBD,
    Zn,
    SmtB_Zn,
    RD_smtO,
    InvR_S,
    AND_Gate,
    InvR_D,
    Gated_TS,
    RQ,
    SD,
    Q,
]

# =============================================================================
# Build CRN
# =============================================================================

crn = ChemicalReactionNetwork(
    species=species,
    reactions=reactions,
)

# =============================================================================
# Output
# =============================================================================

print(crn)

# Optional exports:
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r3.xml')
# crn.write_bioscrape_xml("tetO_smtO_AND_gate_bioscrape.xml")

Species = TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T, SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S, AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
Reactions = [
	TetR+D_tetO <--> TetRD
	TetR+tet <--> TetR_tet
	RNAP+D_tetO <--> RD_tetO
	RD_tetO --> InvR_T+RNAP+D_tetO
	SmtB+D_smtO <--> SmtBD
	SmtB+Zn <--> SmtB_Zn
	RNAP+D_smtO <--> RD_smtO
	RD_smtO --> InvR_S+RNAP+D_smtO
	InvR_T+InvR_S+AND_Gate --> InvR_D+Gated_TS
	InvR_D+RQ --> SD+Q
]


True

### Attempt 4 [Failed]

In [15]:
"""
BioCRNpyler model for:

1. tetO-regulated transcription template
2. smtO-regulated transcription template
3. Strand displacement AND gate
4. Output strand displacement

This script directly implements the reactions given in the specification
using ExplicitSpecies and ChemicalReaction objects.
"""

from biocrnpyler import *

# =============================================================================
# Species
# =============================================================================

# Regulators
TetR = Species("TetR")
tet = Species("tet")
SmtB = Species("SmtB")
Zn = Species("Zn")
RNAP = Species("RNAP")

# DNA templates
D_tetO = Species("D_tetO", material_type="dna")
D_smtO = Species("D_smtO", material_type="dna")

# Complexes
TetRD = Complex([TetR, D_tetO])
TetR_tet = Complex([TetR, tet])

SmtBD = Complex([SmtB, D_smtO])
SmtB_Zn = Complex([SmtB, Zn])

RD_tetO = Complex([RNAP, D_tetO])
RD_smtO = Complex([RNAP, D_smtO])

# RNA products
InvR_T = Species("InvR_T", material_type="rna")
InvR_S = Species("InvR_S", material_type="rna")
InvR_D = Species("InvR_D", material_type="rna")

# Strand displacement species
AND_Gate = Species("AND_Gate")
Gated_TS = Species("Gated_TS")

RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# =============================================================================
# Parameters
# =============================================================================

parameters = {
    # Transcription
    "k_m_tetO": 0.10,
    "k_m_smtO": 0.04,

    # RNAP binding
    "k_bind": 56.0,
    "k_unbind": 0.2,

    # Strand displacement
    "k_SD": 0.04,
    "k_SD_AND": 0.08,

    # TetR repression
    "k_rep_T": 2.98,
    "k_derep_T": 0.001,

    # Tet induction
    "k_ind_T": 4.0,
    "k_unin_T": 0.022,

    # SmtB repression
    "k_rep_S": 3.0,
    "k_derep_S": 0.18,

    # Zn induction
    "k_ind_S": 80.0,
    "k_unin_S": 0.1,
}

# =============================================================================
# Reactions
# =============================================================================

reactions = []

# -------------------------------------------------------------------------
# tetO template
# -------------------------------------------------------------------------

# TetR + D_tetO <-> TetRD
reactions.append(
    ChemicalReaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        propensity_type=MassAction(k_forward=parameters["k_rep_T"])
    )
)

reactions.append(
    ChemicalReaction(
        inputs=[TetRD],
        outputs=[TetR, D_tetO],
        propensity_type=MassAction(k_forward=parameters["k_derep_T"])
    )
)

# TetR + tet <-> TetR_tet
reactions.append(
    ChemicalReaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        propensity_type=MassAction(k_forward=parameters["k_ind_T"])
    )
)

reactions.append(
    ChemicalReaction(
        inputs=[TetR_tet],
        outputs=[TetR, tet],
        propensity_type=MassAction(k_forward=parameters["k_unin_T"])
    )
)

# RNAP + D_tetO <-> RD_tetO
reactions.append(
    ChemicalReaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        propensity_type=MassAction(k_forward=parameters["k_bind"])
    )
)

reactions.append(
    ChemicalReaction(
        inputs=[RD_tetO],
        outputs=[RNAP, D_tetO],
        propensity_type=MassAction(k_forward=parameters["k_unbind"])
    )
)

# Transcription
reactions.append(
    ChemicalReaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        propensity_type=MassAction(k_forward=parameters["k_m_tetO"])
    )
)

# -------------------------------------------------------------------------
# smtO template
# -------------------------------------------------------------------------

# SmtB + D_smtO <-> SmtBD
reactions.append(
    ChemicalReaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        propensity_type=MassAction(k_forward=parameters["k_rep_S"])
    )
)

reactions.append(
    ChemicalReaction(
        inputs=[SmtBD],
        outputs=[SmtB, D_smtO],
        propensity_type=MassAction(k_forward=parameters["k_derep_S"])
    )
)

# SmtB + Zn <-> SmtB_Zn
reactions.append(
    ChemicalReaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        propensity_type=MassAction(k_forward=parameters["k_ind_S"])
    )
)

reactions.append(
    ChemicalReaction(
        inputs=[SmtB_Zn],
        outputs=[SmtB, Zn],
        propensity_type=MassAction(k_forward=parameters["k_unin_S"])
    )
)

# RNAP + D_smtO <-> RD_smtO
reactions.append(
    ChemicalReaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        propensity_type=MassAction(k_forward=parameters["k_bind"])
    )
)

reactions.append(
    ChemicalReaction(
        inputs=[RD_smtO],
        outputs=[RNAP, D_smtO],
        propensity_type=MassAction(k_forward=parameters["k_unbind"])
    )
)

# Transcription
reactions.append(
    ChemicalReaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        propensity_type=MassAction(k_forward=parameters["k_m_smtO"])
    )
)

# -------------------------------------------------------------------------
# Strand displacement
# -------------------------------------------------------------------------

# InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS
reactions.append(
    ChemicalReaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        propensity_type=MassAction(k_forward=parameters["k_SD_AND"])
    )
)

# InvR_D + RQ -> SD + Q
reactions.append(
    ChemicalReaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        propensity_type=MassAction(k_forward=parameters["k_SD"])
    )
)

# =============================================================================
# Build CRN
# =============================================================================

all_species = [
    TetR,
    tet,
    SmtB,
    Zn,
    RNAP,
    D_tetO,
    D_smtO,
    TetRD,
    TetR_tet,
    SmtBD,
    SmtB_Zn,
    RD_tetO,
    RD_smtO,
    InvR_T,
    InvR_S,
    InvR_D,
    AND_Gate,
    Gated_TS,
    RQ,
    SD,
    Q,
]

crn = ChemicalReactionNetwork(
    species=all_species,
    reactions=reactions
)

# =============================================================================
# Output
# =============================================================================

print(crn)

# Optional SBML export:
# from biocrnpyler import CRNLab
# CRNLab().export_sbml_file(crn, "biosensor_and_gate.xml")
# Export SBML
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r4.xml') # export manaully added

NameError: name 'ChemicalReaction' is not defined

### Attempt 5 [Successful]

In [16]:
"""
BioCRNpyler implementation of the CRN described in the prompt.

This script explicitly constructs all species and reactions using
ChemicalReactionNetwork, Species, Reaction, and MassAction kinetics.
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork
from biocrnpyler import MassAction

# =============================================================================
# Species
# =============================================================================

# Proteins / regulators
TetR = Species("TetR", material_type="protein")
SmtB = Species("SmtB", material_type="protein")
RNAP = Species("RNAP", material_type="protein")

# Small molecules
tet = Species("tet", material_type="small_molecule")
Zn = Species("Zn", material_type="small_molecule")

# DNA templates
D_tetO = Species("D_tetO", material_type="dna")
D_smtO = Species("D_smtO", material_type="dna")

# Complexes
TetRD = Species("TetRD", material_type="complex")
TetR_tet = Species("TetR_tet", material_type="complex")
RD_tetO = Species("RD_tetO", material_type="complex")

SmtBD = Species("SmtBD", material_type="complex")
SmtB_Zn = Species("SmtB_Zn", material_type="complex")
RD_smtO = Species("RD_smtO", material_type="complex")

# RNA products
InvR_T = Species("InvR_T", material_type="rna")
InvR_S = Species("InvR_S", material_type="rna")
InvR_D = Species("InvR_D", material_type="rna")

# Strand displacement species
AND_Gate = Species("AND_Gate")
Gated_TS = Species("Gated_TS")

RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# =============================================================================
# Parameters
# =============================================================================

params = {

    # Transcription
    "k_m_tetO": 0.10,
    "k_m_smtO": 0.04,

    # RNAP binding
    "k_bind": 56.0,
    "k_unbind": 0.2,

    # Strand displacement
    "k_SD": 0.04,
    "k_SD_AND": 0.08,

    # TetR repression
    "k_rep_T": 2.98,
    "k_derep_T": 0.001,

    # Tet induction
    "k_ind_T": 4.0,
    "k_unin_T": 0.022,

    # SmtB repression
    "k_rep_S": 3.0,
    "k_derep_S": 0.18,

    # Zn induction
    "k_ind_S": 80.0,
    "k_unin_S": 0.1,
}

# =============================================================================
# Reactions
# =============================================================================

reactions = []

# -------------------------------------------------------------------------
# tetO-regulated template
# -------------------------------------------------------------------------

# TetR + D_tetO <-> TetRD
reactions.append(
    Reaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        propensity_type=MassAction(k_forward=params["k_rep_T"],
                                   k_reverse=params["k_derep_T"])
    )
)

# TetR + tet <-> TetR_tet
reactions.append(
    Reaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        propensity_type=MassAction(k_forward=params["k_ind_T"],
                                   k_reverse=params["k_unin_T"])
    )
)

# RNAP + D_tetO <-> RD_tetO
reactions.append(
    Reaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        propensity_type=MassAction(k_forward=params["k_bind"],
                                   k_reverse=params["k_unbind"])
    )
)

# RD_tetO -> InvR_T + RNAP + D_tetO
reactions.append(
    Reaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        propensity_type=MassAction(k_forward=params["k_m_tetO"])
    )
)

# -------------------------------------------------------------------------
# smtO-regulated template
# -------------------------------------------------------------------------

# SmtB + D_smtO <-> SmtBD
reactions.append(
    Reaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        propensity_type=MassAction(k_forward=params["k_rep_S"],
                                   k_reverse=params["k_derep_S"])
    )
)

# SmtB + Zn <-> SmtB_Zn
reactions.append(
    Reaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        propensity_type=MassAction(k_forward=params["k_ind_S"],
                                   k_reverse=params["k_unin_S"])
    )
)

# RNAP + D_smtO <-> RD_smtO
reactions.append(
    Reaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        propensity_type=MassAction(k_forward=params["k_bind"],
                                   k_reverse=params["k_unbind"])
    )
)

# RD_smtO -> InvR_S + RNAP + D_smtO
reactions.append(
    Reaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        propensity_type=MassAction(k_forward=params["k_m_smtO"])
    )
)

# -------------------------------------------------------------------------
# Strand displacement reactions
# -------------------------------------------------------------------------

# InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS
reactions.append(
    Reaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        propensity_type=MassAction(k_forward=params["k_SD_AND"])
    )
)

# InvR_D + RQ -> SD + Q
reactions.append(
    Reaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        propensity_type=MassAction(k_forward=params["k_SD"])
    )
)

# =============================================================================
# Construct CRN
# =============================================================================

crn = ChemicalReactionNetwork(
    species=[
        TetR, SmtB, RNAP,
        tet, Zn,
        D_tetO, D_smtO,
        TetRD, TetR_tet, RD_tetO,
        SmtBD, SmtB_Zn, RD_smtO,
        InvR_T, InvR_S, InvR_D,
        AND_Gate, Gated_TS,
        RQ, SD, Q
    ],
    reactions=reactions
)

# =============================================================================
# Output
# =============================================================================

print(crn)

# Optional SBML export
# from biocrnpyler import CRNLab
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r5.xml')

Species = protein_TetR, protein_SmtB, protein_RNAP, small_molecule_tet, small_molecule_Zn, dna_D_tetO, dna_D_smtO, complex_TetRD, complex_TetR_tet, complex_RD_tetO, complex_SmtBD, complex_SmtB_Zn, complex_RD_smtO, rna_InvR_T, rna_InvR_S, rna_InvR_D, AND_Gate, Gated_TS, RQ, SD, Q
Reactions = [
	protein[TetR]+dna[D_tetO] <--> complex[TetRD]
	protein[TetR]+small_molecule[tet] <--> complex[TetR_tet]
	protein[RNAP]+dna[D_tetO] <--> complex[RD_tetO]
	complex[RD_tetO] --> rna[InvR_T]+protein[RNAP]+dna[D_tetO]
	protein[SmtB]+dna[D_smtO] <--> complex[SmtBD]
	protein[SmtB]+small_molecule[Zn] <--> complex[SmtB_Zn]
	protein[RNAP]+dna[D_smtO] <--> complex[RD_smtO]
	complex[RD_smtO] --> rna[InvR_S]+protein[RNAP]+dna[D_smtO]
	rna[InvR_T]+rna[InvR_S]+AND_Gate --> rna[InvR_D]+Gated_TS
	rna[InvR_D]+RQ --> SD+Q
]


True

### Attempt 6 [Failed]

In [17]:
from biocrnpyler import (
    Species,
    Reaction,
    ChemicalReactionNetwork,
    Parameter
)

###############################################################################
# Species
###############################################################################

# Proteins / Regulators
TetR = Species("TetR")
SmtB = Species("SmtB")
RNAP = Species("RNAP")

# Small molecules
tet = Species("tet")
Zn = Species("Zn")

# DNA templates
D_tetO = Species("D_tetO")
D_smtO = Species("D_smtO")

# DNA complexes
TetRD = Species("TetRD")
SmtBD = Species("SmtBD")

RD_tetO = Species("RD_tetO")
RD_smtO = Species("RD_smtO")

TetR_tet = Species("TetR_tet")
SmtB_Zn = Species("SmtB_Zn")

# RNA species
InvR_T = Species("InvR_T")
InvR_S = Species("InvR_S")
InvR_D = Species("InvR_D")

# Strand displacement species
AND_Gate = Species("AND_Gate")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

###############################################################################
# Parameters
###############################################################################

k_m_tetO   = Parameter("k_m_tetO", 0.10)
k_m_smtO   = Parameter("k_m_smtO", 0.04)

k_bind     = Parameter("k_bind", 56.0)
k_unbind   = Parameter("k_unbind", 0.20)

k_SD       = Parameter("k_SD", 0.04)
k_SD_AND   = Parameter("k_SD_AND", 0.08)

k_rep_T    = Parameter("k_rep_T", 2.98)
k_derep_T  = Parameter("k_derep_T", 0.001)

k_ind_T    = Parameter("k_ind_T", 4.0)
k_unin_T   = Parameter("k_unin_T", 0.022)

k_rep_S    = Parameter("k_rep_S", 3.0)
k_derep_S  = Parameter("k_derep_S", 0.18)

k_ind_S    = Parameter("k_ind_S", 80.0)
k_unin_S   = Parameter("k_unin_S", 0.10)

###############################################################################
# Reactions
###############################################################################

reactions = []

###############################################################################
# tetO-regulated template
###############################################################################

# TetR + D_tetO <-> TetRD
reactions.append(
    Reaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=k_rep_T,
        k_reverse=k_derep_T
    )
)

# TetR + tet <-> TetR_tet
reactions.append(
    Reaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=k_ind_T,
        k_reverse=k_unin_T
    )
)

# RNAP + D_tetO <-> RD_tetO
reactions.append(
    Reaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=k_bind,
        k_reverse=k_unbind
    )
)

# RD_tetO -> InvR_T + RNAP + D_tetO
reactions.append(
    Reaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO
    )
)

###############################################################################
# smtO-regulated template
###############################################################################

# SmtB + D_smtO <-> SmtBD
reactions.append(
    Reaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=k_rep_S,
        k_reverse=k_derep_S
    )
)

# SmtB + Zn <-> SmtB_Zn
reactions.append(
    Reaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=k_ind_S,
        k_reverse=k_unin_S
    )
)

# RNAP + D_smtO <-> RD_smtO
reactions.append(
    Reaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=k_bind,
        k_reverse=k_unbind
    )
)

# RD_smtO -> InvR_S + RNAP + D_smtO
reactions.append(
    Reaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO
    )
)

###############################################################################
# Strand displacement reactions
###############################################################################

# InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS
reactions.append(
    Reaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND
    )
)

# InvR_D + RQ -> SD + Q
reactions.append(
    Reaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=k_SD
    )
)

###############################################################################
# Build CRN
###############################################################################

crn = ChemicalReactionNetwork(reactions=reactions)

###############################################################################
# Display
###############################################################################

print(crn)

# Optional:
# Export SBML
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r6.xml')

TypeError: Reaction.__init__() got an unexpected keyword argument 'k_forward'

### Attempt 7 [Successful]

In [18]:
"""
BioCRNpyler implementation of the tetO/smtO regulated strand displacement network.

This script explicitly constructs the CRN from the reactions given in the prompt,
using Species and Reaction objects rather than high-level Components.
"""

from biocrnpyler import *

# =============================================================================
# Species
# =============================================================================

# Proteins / transcription factors
TetR = Species("TetR", material_type="protein")
SmtB = Species("SmtB", material_type="protein")
RNAP = Species("RNAP", material_type="protein")

# Small molecules
tet = Species("tet")
Zn = Species("Zn")

# DNA templates
D_tetO = Species("D_tetO", material_type="dna")
D_smtO = Species("D_smtO", material_type="dna")

# Complexes
TetRD = Complex([TetR, D_tetO])
TetR_tet = Complex([TetR, tet])

SmtBD = Complex([SmtB, D_smtO])
SmtB_Zn = Complex([SmtB, Zn])

RD_tetO = Complex([RNAP, D_tetO])
RD_smtO = Complex([RNAP, D_smtO])

# RNA outputs
InvR_T = Species("InvR_T", material_type="rna")
InvR_S = Species("InvR_S", material_type="rna")
InvR_D = Species("InvR_D", material_type="rna")

# Strand displacement species
AND_Gate = Species("AND_Gate")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

# =============================================================================
# Parameters
# =============================================================================

parameters = {
    # Transcription
    "k_m_tetO": 0.10,
    "k_m_smtO": 0.04,

    # RNAP binding
    "k_bind": 56.0,
    "k_unbind": 0.20,

    # Strand displacement
    "k_SD": 0.04,
    "k_SD_AND": 0.08,

    # TetR repression
    "k_rep_T": 2.98,
    "k_derep_T": 0.001,

    # Tet induction
    "k_ind_T": 4.0,
    "k_unin_T": 0.022,

    # SmtB repression
    "k_rep_S": 3.0,
    "k_derep_S": 0.18,

    # Zn induction
    "k_ind_S": 80.0,
    "k_unin_S": 0.10,
}

# =============================================================================
# Reactions
# =============================================================================

reactions = [

    # -------------------------------------------------------------------------
    # tetO-regulated template
    # -------------------------------------------------------------------------

    # TetR + D_tetO <-> TetRD
    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=parameters["k_rep_T"],
        k_reverse=parameters["k_derep_T"],
    ),

    # TetR + tet <-> TetR-tet
    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=parameters["k_ind_T"],
        k_reverse=parameters["k_unin_T"],
    ),

    # RNAP + D_tetO <-> RD_tetO
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=parameters["k_bind"],
        k_reverse=parameters["k_unbind"],
    ),

    # RD_tetO -> InvR_T + RNAP + D_tetO
    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=parameters["k_m_tetO"],
    ),

    # -------------------------------------------------------------------------
    # smtO-regulated template
    # -------------------------------------------------------------------------

    # SmtB + D_smtO <-> SmtBD
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=parameters["k_rep_S"],
        k_reverse=parameters["k_derep_S"],
    ),

    # SmtB + Zn <-> SmtB-Zn
    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=parameters["k_ind_S"],
        k_reverse=parameters["k_unin_S"],
    ),

    # RNAP + D_smtO <-> RD_smtO
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=parameters["k_bind"],
        k_reverse=parameters["k_unbind"],
    ),

    # RD_smtO -> InvR_S + RNAP + D_smtO
    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=parameters["k_m_smtO"],
    ),

    # -------------------------------------------------------------------------
    # Strand displacement reactions
    # -------------------------------------------------------------------------

    # InvR_T + InvR_S + AND-Gate -> InvR_D + Gated_TS
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=parameters["k_SD_AND"],
    ),

    # InvR_D + RQ -> SD + Q
    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=parameters["k_SD"],
    ),
]

# =============================================================================
# Build CRN
# =============================================================================

crn = ChemicalReactionNetwork(
    species=[
        TetR,
        SmtB,
        RNAP,
        tet,
        Zn,
        D_tetO,
        D_smtO,
        TetRD,
        TetR_tet,
        SmtBD,
        SmtB_Zn,
        RD_tetO,
        RD_smtO,
        InvR_T,
        InvR_S,
        InvR_D,
        AND_Gate,
        Gated_TS,
        RQ,
        SD,
        Q,
    ],
    reactions=reactions,
)

# =============================================================================
# Display summary
# =============================================================================

print("Species:", len(crn.species))
print("Reactions:", len(crn.reactions))

for r in crn.reactions:
    print(r)

# Export SBML
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r7.xml') # export manaully added

Species: 21
Reactions: 10
protein[TetR]+dna[D_tetO] <--> complex[dna[D_tetO]:protein[TetR]]
protein[TetR]+tet <--> complex[protein[TetR]:tet]
protein[RNAP]+dna[D_tetO] <--> complex[dna[D_tetO]:protein[RNAP]]
complex[dna[D_tetO]:protein[RNAP]] --> rna[InvR_T]+protein[RNAP]+dna[D_tetO]
protein[SmtB]+dna[D_smtO] <--> complex[dna[D_smtO]:protein[SmtB]]
protein[SmtB]+Zn <--> complex[Zn:protein[SmtB]]
protein[RNAP]+dna[D_smtO] <--> complex[dna[D_smtO]:protein[RNAP]]
complex[dna[D_smtO]:protein[RNAP]] --> rna[InvR_S]+protein[RNAP]+dna[D_smtO]
rna[InvR_T]+rna[InvR_S]+AND_Gate --> rna[InvR_D]+Gated_TS
rna[InvR_D]+RQ --> SD+Q


True

### Attempt 8 [Failed]

In [19]:
from biocrnpyler import *

# ============================================================
# Species
# ============================================================

TetR      = Species("TetR")
tet       = Species("tet")
D_tetO    = Species("D_tetO")
TetRD     = Species("TetRD")
TetR_tet  = Species("TetR_tet")
RNAP      = Species("RNAP")
RD_tetO   = Species("RD_tetO")
InvR_T    = Species("InvR_T")

SmtB      = Species("SmtB")
Zn        = Species("Zn")
D_smtO    = Species("D_smtO")
SmtBD     = Species("SmtBD")
SmtB_Zn   = Species("SmtB_Zn")
RD_smtO   = Species("RD_smtO")
InvR_S    = Species("InvR_S")

AND_Gate  = Species("AND_Gate")
InvR_D    = Species("InvR_D")
Gated_TS  = Species("Gated_TS")

RQ        = Species("RQ")
SD        = Species("SD")
Q         = Species("Q")

# ============================================================
# Parameters
# ============================================================

params = ParameterDatabase(parameter_dictionary={
    ("k_forward", "tetR_bind"):      2.98,
    ("k_reverse", "tetR_bind"):      0.001,

    ("k_forward", "tet_induction"):  4.0,
    ("k_reverse", "tet_induction"):  0.022,

    ("k_forward", "smtB_bind"):      3.0,
    ("k_reverse", "smtB_bind"):      0.18,

    ("k_forward", "zn_bind"):        80.0,
    ("k_reverse", "zn_bind"):        0.10,

    ("k_forward", "rnap_bind"):      56.0,
    ("k_reverse", "rnap_bind"):      0.20,

    ("k_cat", "tx_tetO"):            0.10,
    ("k_cat", "tx_smtO"):            0.04,

    ("k_cat", "strand_displacement"): 0.04,
    ("k_cat", "and_gate"):            0.08,
})

# ============================================================
# Reactions
# ============================================================

reactions = [

    # --------------------------------------------------------
    # tetO template regulation
    # --------------------------------------------------------

    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=2.98,
        k_reverse=0.001
    ),

    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=4.0,
        k_reverse=0.022
    ),

    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=56.0,
        k_reverse=0.2
    ),

    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=0.10
    ),

    # --------------------------------------------------------
    # smtO template regulation
    # --------------------------------------------------------

    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=3.0,
        k_reverse=0.18
    ),

    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=80.0,
        k_reverse=0.10
    ),

    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=56.0,
        k_reverse=0.20
    ),

    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=0.04
    ),

    # --------------------------------------------------------
    # Strand displacement AND gate
    # --------------------------------------------------------

    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=0.08
    ),

    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=0.04
    ),
]

# ============================================================
# Build CRN
# ============================================================

crn = ChemicalReactionNetwork(
    species=[
        TetR, tet, D_tetO, TetRD, TetR_tet,
        RNAP, RD_tetO, InvR_T,
        SmtB, Zn, D_smtO, SmtBD, SmtB_Zn,
        RD_smtO, InvR_S,
        AND_Gate, InvR_D, Gated_TS,
        RQ, SD, Q
    ],
    reactions=reactions
)

# ============================================================
# Output
# ============================================================

print(crn)

# Export SBML
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r8.xml')

ValueError: parameter_key must be None, a dictionary, a ParameterKey, a 3-tuple, or a string (parameter name): received ('k_forward', 'tetR_bind').

### Attempt 9 [Failed]

In [20]:
from biocrnpyler import (
    Species,
    Reaction,
    ChemicalReactionNetwork,
    Parameter
)

###############################################################################
# Species
###############################################################################

# Proteins / regulators
TetR = Species("TetR", material_type="protein")
SmtB = Species("SmtB", material_type="protein")
RNAP = Species("RNAP", material_type="protein")

# Small molecules
tet = Species("tet")
Zn = Species("Zn")

# DNA templates
D_tetO = Species("D_tetO", material_type="dna")
D_smtO = Species("D_smtO", material_type="dna")

# Complexes
TetRD = Species("TetRD")
TetR_tet = Species("TetR_tet")
RD_tetO = Species("RD_tetO")

SmtBD = Species("SmtBD")
SmtB_Zn = Species("SmtB_Zn")
RD_smtO = Species("RD_smtO")

# RNA outputs
InvR_T = Species("InvR_T", material_type="rna")
InvR_S = Species("InvR_S", material_type="rna")
InvR_D = Species("InvR_D", material_type="rna")

# Strand displacement species
AND_Gate = Species("AND_Gate")
Gated_TS = Species("Gated_TS")

RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

###############################################################################
# Parameters
###############################################################################

k_m_tetO = Parameter("k_m_tetO", 0.10)
k_m_smtO = Parameter("k_m_smtO", 0.04)

k_bind = Parameter("k_bind", 56.0)
k_unbind = Parameter("k_unbind", 0.20)

k_SD = Parameter("k_SD", 0.04)
k_SD_AND = Parameter("k_SD_AND", 0.08)

k_rep_T = Parameter("k_rep_T", 2.98)
k_derep_T = Parameter("k_derep_T", 0.001)

k_ind_T = Parameter("k_ind_T", 4.0)
k_unin_T = Parameter("k_unin_T", 0.022)

k_rep_S = Parameter("k_rep_S", 3.0)
k_derep_S = Parameter("k_derep_S", 0.18)

k_ind_S = Parameter("k_ind_S", 80.0)
k_unin_S = Parameter("k_unin_S", 0.10)

###############################################################################
# Reactions
###############################################################################

reactions = [

    ###########################################################################
    # tetO-regulated template
    ###########################################################################

    # TetR + D_tetO <-> TetRD
    Reaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=k_rep_T,
        k_reverse=k_derep_T
    ),

    # TetR + tet <-> TetR_tet
    Reaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=k_ind_T,
        k_reverse=k_unin_T
    ),

    # RNAP + D_tetO <-> RD_tetO
    Reaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=k_bind,
        k_reverse=k_unbind
    ),

    # RD_tetO -> InvR_T + RNAP + D_tetO
    Reaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO
    ),

    ###########################################################################
    # smtO-regulated template
    ###########################################################################

    # SmtB + D_smtO <-> SmtBD
    Reaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=k_rep_S,
        k_reverse=k_derep_S
    ),

    # SmtB + Zn <-> SmtB_Zn
    Reaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=k_ind_S,
        k_reverse=k_unin_S
    ),

    # RNAP + D_smtO <-> RD_smtO
    Reaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=k_bind,
        k_reverse=k_unbind
    ),

    # RD_smtO -> InvR_S + RNAP + D_smtO
    Reaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO
    ),

    ###########################################################################
    # Strand displacement
    ###########################################################################

    # InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS
    Reaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND
    ),

    # InvR_D + RQ -> SD + Q
    Reaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=k_SD
    )

]

###############################################################################
# Build the CRN
###############################################################################

crn = ChemicalReactionNetwork(
    species=[
        TetR,
        SmtB,
        RNAP,
        tet,
        Zn,
        D_tetO,
        D_smtO,
        TetRD,
        TetR_tet,
        RD_tetO,
        SmtBD,
        SmtB_Zn,
        RD_smtO,
        InvR_T,
        InvR_S,
        InvR_D,
        AND_Gate,
        Gated_TS,
        RQ,
        SD,
        Q
    ],
    reactions=reactions
)

###############################################################################
# Display CRN
###############################################################################

print(crn)

# Optional:
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r9.xml')

TypeError: Reaction.__init__() got an unexpected keyword argument 'k_forward'

### Attempt 10 [Failed]

In [21]:
from biocrnpyler import *

# ============================================================
# Parameters
# ============================================================

parameters = {
    # tetO template
    "k_rep_T": 2.98,
    "k_derep_T": 0.001,
    "k_ind_T": 4.0,
    "k_unin_T": 0.022,

    # smtO template
    "k_rep_S": 3.0,
    "k_derep_S": 0.18,
    "k_ind_S": 80.0,
    "k_unin_S": 0.1,

    # transcription
    "k_bind": 56.0,
    "k_unbind": 0.2,
    "k_m_tetO": 0.1,
    "k_m_smtO": 0.04,

    # strand displacement
    "k_SD": 0.04,
    "k_SD_AND": 0.08,
}

# ============================================================
# Species
# ============================================================

species_names = [
    # tetO regulation
    "TetR",
    "D_tetO",
    "TetRD",
    "tet",
    "TetR_tet",

    # smtO regulation
    "SmtB",
    "D_smtO",
    "SmtBD",
    "Zn",
    "SmtB_Zn",

    # RNAP binding
    "RNAP",
    "RD_tetO",
    "RD_smtO",

    # outputs
    "InvR_T",
    "InvR_S",

    # strand displacement
    "AND_Gate",
    "InvR_D",
    "Gated_TS",
    "RQ",
    "SD",
    "Q",
]

species = {
    name: Species(name)
    for name in species_names
}

# ============================================================
# Reactions
# ============================================================

reactions = []

# ------------------------------------------------------------
# TetO-regulated Template
# ------------------------------------------------------------

# TetR + D_tetO <--> TetRD
reactions.append(
    Reaction(
        [species["TetR"], species["D_tetO"]],
        [species["TetRD"]],
        k_forward=parameters["k_rep_T"],
        k_reverse=parameters["k_derep_T"]
    )
)

# TetR + tet <--> TetR-tet
reactions.append(
    Reaction(
        [species["TetR"], species["tet"]],
        [species["TetR_tet"]],
        k_forward=parameters["k_ind_T"],
        k_reverse=parameters["k_unin_T"]
    )
)

# RNAP + D_tetO <--> RD_tetO
reactions.append(
    Reaction(
        [species["RNAP"], species["D_tetO"]],
        [species["RD_tetO"]],
        k_forward=parameters["k_bind"],
        k_reverse=parameters["k_unbind"]
    )
)

# RD_tetO --> InvR_T + RNAP + D_tetO
reactions.append(
    Reaction(
        [species["RD_tetO"]],
        [species["InvR_T"],
         species["RNAP"],
         species["D_tetO"]],
        k_forward=parameters["k_m_tetO"]
    )
)

# ------------------------------------------------------------
# smtO-regulated Template
# ------------------------------------------------------------

# SmtB + D_smtO <--> SmtBD
reactions.append(
    Reaction(
        [species["SmtB"], species["D_smtO"]],
        [species["SmtBD"]],
        k_forward=parameters["k_rep_S"],
        k_reverse=parameters["k_derep_S"]
    )
)

# SmtB + Zn <--> SmtB-Zn
reactions.append(
    Reaction(
        [species["SmtB"], species["Zn"]],
        [species["SmtB_Zn"]],
        k_forward=parameters["k_ind_S"],
        k_reverse=parameters["k_unin_S"]
    )
)

# RNAP + D_smtO <--> RD_smtO
reactions.append(
    Reaction(
        [species["RNAP"], species["D_smtO"]],
        [species["RD_smtO"]],
        k_forward=parameters["k_bind"],
        k_reverse=parameters["k_unbind"]
    )
)

# RD_smtO --> InvR_S + RNAP + D_smtO
reactions.append(
    Reaction(
        [species["RD_smtO"]],
        [species["InvR_S"],
         species["RNAP"],
         species["D_smtO"]],
        k_forward=parameters["k_m_smtO"]
    )
)

# ------------------------------------------------------------
# Strand displacement reactions
# ------------------------------------------------------------

# InvR_T + InvR_S + AND-Gate --> InvR_D + Gated_TS
reactions.append(
    Reaction(
        [
            species["InvR_T"],
            species["InvR_S"],
            species["AND_Gate"]
        ],
        [
            species["InvR_D"],
            species["Gated_TS"]
        ],
        k_forward=parameters["k_SD_AND"]
    )
)

# InvR_D + RQ --> SD + Q
reactions.append(
    Reaction(
        [
            species["InvR_D"],
            species["RQ"]
        ],
        [
            species["SD"],
            species["Q"]
        ],
        k_forward=parameters["k_SD"]
    )
)

# ============================================================
# Build CRN
# ============================================================

crn = ChemicalReactionNetwork(
    species=list(species.values()),
    reactions=reactions
)

# ============================================================
# Print CRN
# ============================================================

print(crn.pretty_print())

# Optional: export SBML
crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_chatgpt_r10.xml')

TypeError: Reaction.__init__() got an unexpected keyword argument 'k_forward'

## Claude [10/10 Successful]

### Attempt 1 [Successful]

In [22]:
from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

def build_crn():
    # ==========================================
    # 1. Define the Species
    # ==========================================
    
    # Global Species
    RNAP = Species("RNAP")
    
    # tetO-regulated Template Species
    TetR     = Species("TetR")
    D_tetO   = Species("D_tetO")
    TetRD    = Species("TetRD")
    tet      = Species("tet")
    TetR_tet = Species("TetR_tet")  # Hyphen replaced with underscore for valid Python variable
    RD_tetO  = Species("RD_tetO")
    InvR_T   = Species("InvR_T")
    
    # smtO-Regulated Template Species
    SmtB     = Species("SmtB")
    D_smtO   = Species("D_smtO")
    SmtBD    = Species("SmtBD")
    Zn       = Species("Zn")
    SmtB_Zn  = Species("SmtB_Zn")
    RD_smtO  = Species("RD_smtO")
    InvR_S   = Species("InvR_S")
    
    # Strand Displacements Species
    AND_Gate = Species("AND_Gate") 
    InvR_D   = Species("InvR_D")
    Gated_TS = Species("Gated_TS")
    RQ       = Species("RQ")
    SD       = Species("SD")
    Q        = Species("Q")
    
    # Combine into a single list
    species_list = [
        RNAP, TetR, D_tetO, TetRD, tet, TetR_tet, RD_tetO, InvR_T,
        SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
        AND_Gate, InvR_D, Gated_TS, RQ, SD, Q
    ]
    
    # ==========================================
    # 2. Define Parameters 
    # ==========================================
    
    # Transcription / Binding Rates
    k_m_tetO = 0.1
    k_m_smtO = 0.04
    k_bind   = 56
    k_unbind = 0.2
    
    # Strand Displacement Rates
    k_SD     = 0.04
    k_SD_AND = 0.08
    
    # tetO Repression / Induction Rates
    k_rep_T   = 2.98
    k_derep_T = 0.001
    k_ind_T   = 4.0
    k_unin_T  = 0.022
    
    # smtO Repression / Induction Rates
    k_rep_S   = 3.0
    k_derep_S = 0.18
    k_ind_S   = 80
    k_unin_S  = 0.1

    # ==========================================
    # 3. Build the Reactions
    # ==========================================
    reactions = []
    
    # --- tetO-regulated Template ---
    # TetR + D_tetO <---> TetRD
    reactions.append(Reaction.from_massaction([TetR, D_tetO], [TetRD], k_forward=k_rep_T, k_reverse=k_derep_T))
    
    # TetR + tet <---> TetR-tet
    reactions.append(Reaction.from_massaction([TetR, tet], [TetR_tet], k_forward=k_ind_T, k_reverse=k_unin_T))
    
    # RNAP + D_tetO <---> RD_tetO
    reactions.append(Reaction.from_massaction([RNAP, D_tetO], [RD_tetO], k_forward=k_bind, k_reverse=k_unbind))
    
    # RD_tetO ---> InvR_T + RNAP + D_tetO
    reactions.append(Reaction.from_massaction([RD_tetO], [InvR_T, RNAP, D_tetO], k_forward=k_m_tetO))
    
    
    # --- smtO-Regulated Template ---
    # SmtB + D_smtO <---> SmtBD
    reactions.append(Reaction.from_massaction([SmtB, D_smtO], [SmtBD], k_forward=k_rep_S, k_reverse=k_derep_S))
    
    # SmtB + Zn <---> SmtB-Zn
    reactions.append(Reaction.from_massaction([SmtB, Zn], [SmtB_Zn], k_forward=k_ind_S, k_reverse=k_unin_S))
    
    # RNAP + D_smtO <---> RD_smtO
    reactions.append(Reaction.from_massaction([RNAP, D_smtO], [RD_smtO], k_forward=k_bind, k_reverse=k_unbind))
    
    # RD_smtO ---> InvR_S + RNAP + D_smtO
    reactions.append(Reaction.from_massaction([RD_smtO], [InvR_S, RNAP, D_smtO], k_forward=k_m_smtO))
    
    
    # --- Strand Displacements ---
    # InvR_T + InvR_S + AND-Gate ---> InvR_D + Gated_TS
    reactions.append(Reaction.from_massaction([InvR_T, InvR_S, AND_Gate], [InvR_D, Gated_TS], k_forward=k_SD_AND))
    
    # InvR_D + RQ ---> SD + Q
    reactions.append(Reaction.from_massaction([InvR_D, RQ], [SD, Q], k_forward=k_SD))

    # ==========================================
    # 4. Compile the CRN
    # ==========================================
    crn = ChemicalReactionNetwork(species=species_list, reactions=reactions)
    return crn

if __name__ == "__main__":
    my_crn = build_crn()
    
    # Print a summary of the network
    print(my_crn.pretty_print())

# Export SBML
my_crn.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r1.xml') # export manually added

Species(N = 21) = {
tet (@ 0),  Zn (@ 0),  TetR_tet (@ 0),  TetRD (@ 0),  TetR (@ 0),  SmtB_Zn (@ 0),  SmtBD (@ 0),  SmtB (@ 0),  SD (@ 0),  RQ (@ 0),  RNAP (@ 0),  RD_tetO (@ 0),  RD_smtO (@ 0),  Q (@ 0),  InvR_T (@ 0),  InvR_S (@ 0),  InvR_D (@ 0),  Gated_TS (@ 0),  D_tetO (@ 0),  D_smtO (@ 0),  AND_Gate (@ 0),  
}

Reactions (10) = [
0. TetR+D_tetO <--> TetRD
 Kf=k_forward * TetR * D_tetO
 Kr=k_reverse * TetRD
  k_forward=2.98
  k_reverse=0.001

1. TetR+tet <--> TetR_tet
 Kf=k_forward * TetR * tet
 Kr=k_reverse * TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. RNAP+D_tetO <--> RD_tetO
 Kf=k_forward * RNAP * D_tetO
 Kr=k_reverse * RD_tetO
  k_forward=56
  k_reverse=0.2

3. RD_tetO --> InvR_T+RNAP+D_tetO
 Kf=k_forward * RD_tetO
  k_forward=0.1

4. SmtB+D_smtO <--> SmtBD
 Kf=k_forward * SmtB * D_smtO
 Kr=k_reverse * SmtBD
  k_forward=3.0
  k_reverse=0.18

5. SmtB+Zn <--> SmtB_Zn
 Kf=k_forward * SmtB * Zn
 Kr=k_reverse * SmtB_Zn
  k_forward=80
  k_reverse=0.1

6. RNAP+D_smtO <--> RD_smtO

True

### Attempt 2 [Successful]

In [23]:
"""
BioCRNpyler script: tetO/smtO dual-input transcriptional AND-gate
coupled to a downstream strand-displacement (SD) cascade.

Reaction network summary
-------------------------
tetO-regulated template
    TetR   + D_tetO  <-> TetRD           (k_rep_T   / k_derep_T)
    TetR   + tet     <-> TetR_tet        (k_ind_T   / k_unin_T)
    RNAP   + D_tetO  <-> RD_tetO         (k_bind    / k_unbind)
    RD_tetO -> InvR_T + RNAP + D_tetO    (k_m_tetO)

smtO-regulated template
    SmtB   + D_smtO  <-> SmtBD           (k_rep_S   / k_derep_S)
    SmtB   + Zn      <-> SmtB_Zn         (k_ind_S   / k_unin_S)
    RNAP   + D_smtO  <-> RD_smtO         (k_bind    / k_unbind)
    RD_smtO -> InvR_S + RNAP + D_smtO    (k_m_smtO)

Strand displacement cascade
    InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS   (k_SD_AND)
    InvR_D + RQ                -> SD     + Q          (k_SD)

All reactions are simple mass-action reactions built directly from
biocrnpyler.Species / biocrnpyler.Reaction objects (no Mixture/
Component machinery is needed since these are already elementary
CRN reactions with explicit rate constants).
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ---------------------------------------------------------------------------
# 1. Rate constants (all concentrations assumed in uM, time in seconds)
# ---------------------------------------------------------------------------

# Transcription (isomerization / open-complex escape) rates
k_m_tetO = 0.1     # /sec
k_m_smtO = 0.04    # /sec

# RNAP binding to either promoter (shared)
k_bind = 56        # /uM-sec
k_unbind = 0.2      # /sec

# Strand displacement rates
k_SD = 0.04        # /uM-sec
k_SD_AND = 0.08    # /uM-sec

# TetR repressor / inducer (tet) kinetics
k_rep_T = 2.98     # /uM-sec
k_derep_T = 0.001  # /uM-sec
k_ind_T = 4.0      # /uM-sec
k_unin_T = 0.022   # /sec

# SmtB repressor / inducer (Zn) kinetics
k_rep_S = 3.0      # /uM-sec
k_derep_S = 0.18   # /uM-sec
k_ind_S = 80       # /uM-sec
k_unin_S = 0.1     # /sec

# ---------------------------------------------------------------------------
# 2. Species definitions
# ---------------------------------------------------------------------------

# --- tetO branch ---
TetR = Species("TetR")           # TetR repressor protein
D_tetO = Species("D_tetO")       # free tetO-regulated DNA template
TetRD = Species("TetRD")         # TetR bound (repressed) DNA
tet = Species("tet")             # tetracycline inducer
TetR_tet = Species("TetR_tet")   # TetR sequestered by tetracycline
RD_tetO = Species("RD_tetO")     # RNAP-bound tetO template (open complex)
InvR_T = Species("InvR_T")       # invader strand produced from tetO template

# --- smtO branch ---
SmtB = Species("SmtB")           # SmtB repressor protein
D_smtO = Species("D_smtO")       # free smtO-regulated DNA template
SmtBD = Species("SmtBD")         # SmtB bound (repressed) DNA
Zn = Species("Zn")               # zinc inducer
SmtB_Zn = Species("SmtB_Zn")     # SmtB sequestered by Zn
RD_smtO = Species("RD_smtO")     # RNAP-bound smtO template (open complex)
InvR_S = Species("InvR_S")       # invader strand produced from smtO template

# --- shared ---
RNAP = Species("RNAP")           # RNA polymerase, shared by both templates

# --- strand-displacement cascade ---
AND_Gate = Species("AND_Gate")   # two-input AND-gate strand-displacement complex
InvR_D = Species("InvR_D")       # output invader strand from AND-gate reaction
Gated_TS = Species("Gated_TS")   # waste/byproduct complex from AND-gate reaction
RQ = Species("RQ")               # reporter complex (quenched)
SD = Species("SD")               # displaced strand (signal) product
Q = Species("Q")                 # released quencher/reporter output

# ---------------------------------------------------------------------------
# 3. Reactions
# ---------------------------------------------------------------------------

reactions = []

# ---- tetO-regulated template ----

# TetR + D_tetO <--> TetRD   (repression / de-repression)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=k_rep_T,
        k_reverse=k_derep_T,
    )
)

# TetR + tet <--> TetR_tet   (induction / un-induction)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=k_ind_T,
        k_reverse=k_unin_T,
    )
)

# RNAP + D_tetO <--> RD_tetO   (RNAP binding / unbinding)
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_tetO -> InvR_T + RNAP + D_tetO   (transcription / isomerization)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO,
    )
)

# ---- smtO-regulated template ----

# SmtB + D_smtO <--> SmtBD   (repression / de-repression)
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=k_rep_S,
        k_reverse=k_derep_S,
    )
)

# SmtB + Zn <--> SmtB_Zn   (induction / un-induction)
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=k_ind_S,
        k_reverse=k_unin_S,
    )
)

# RNAP + D_smtO <--> RD_smtO   (RNAP binding / unbinding)
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_smtO -> InvR_S + RNAP + D_smtO   (transcription / isomerization)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO,
    )
)

# ---- Strand displacement cascade (AND-gate logic) ----

# InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND,
    )
)

# InvR_D + RQ -> SD + Q
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=k_SD,
    )
)

# ---------------------------------------------------------------------------
# 4. Assemble the Chemical Reaction Network
# ---------------------------------------------------------------------------

species = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    RNAP,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

CRN = ChemicalReactionNetwork(species=species, reactions=reactions)

if __name__ == "__main__":
    # Print a human-readable summary of the network
    print(CRN.pretty_print())

    # Export as an SBML model for use in simulators (COPASI, Tellurium, etc.)
    CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r2.xml')
    print("\nSBML model written to and_gate_crn.xml")


Species(N = 21) = {
tet (@ 0),  Zn (@ 0),  TetR_tet (@ 0),  TetRD (@ 0),  TetR (@ 0),  SmtB_Zn (@ 0),  SmtBD (@ 0),  SmtB (@ 0),  SD (@ 0),  RQ (@ 0),  RNAP (@ 0),  RD_tetO (@ 0),  RD_smtO (@ 0),  Q (@ 0),  InvR_T (@ 0),  InvR_S (@ 0),  InvR_D (@ 0),  Gated_TS (@ 0),  D_tetO (@ 0),  D_smtO (@ 0),  AND_Gate (@ 0),  
}

Reactions (10) = [
0. TetR+D_tetO <--> TetRD
 Kf=k_forward * TetR * D_tetO
 Kr=k_reverse * TetRD
  k_forward=2.98
  k_reverse=0.001

1. TetR+tet <--> TetR_tet
 Kf=k_forward * TetR * tet
 Kr=k_reverse * TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. RNAP+D_tetO <--> RD_tetO
 Kf=k_forward * RNAP * D_tetO
 Kr=k_reverse * RD_tetO
  k_forward=56
  k_reverse=0.2

3. RD_tetO --> InvR_T+RNAP+D_tetO
 Kf=k_forward * RD_tetO
  k_forward=0.1

4. SmtB+D_smtO <--> SmtBD
 Kf=k_forward * SmtB * D_smtO
 Kr=k_reverse * SmtBD
  k_forward=3.0
  k_reverse=0.18

5. SmtB+Zn <--> SmtB_Zn
 Kf=k_forward * SmtB * Zn
 Kr=k_reverse * SmtB_Zn
  k_forward=80
  k_reverse=0.1

6. RNAP+D_smtO <--> RD_smtO

### Attempt 3 [Successful]

In [24]:
"""
BioCRNpyler script for the tetO/smtO-regulated AND-gate strand-displacement
Chemical Reaction Network (CRN).

Reaction modules:
  1. tetO-regulated template   (TetR repression / tet induction / transcription)
  2. smtO-regulated template   (SmtB repression / Zn induction / transcription)
  3. Strand-displacement AND gate + reporter release

Units: concentrations in uM, time in seconds (rate constants given accordingly:
bimolecular rates in /uM-sec, unimolecular rates in /sec).
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ---------------------------------------------------------------------------
# 1. Rate constants (as given in the problem statement)
# ---------------------------------------------------------------------------

# Transcription (catalytic, unimolecular) rates
k_m_tetO = 0.1      # /sec
k_m_smtO = 0.04     # /sec

# RNAP binding / unbinding to either promoter (shared)
k_bind = 56         # /uM-sec
k_unbind = 0.2       # /sec

# Strand displacement rates
k_SD = 0.04         # /uM-sec
k_SD_AND = 0.08     # /uM-sec

# TetR repression / induction
k_rep_T = 2.98      # /uM-sec
k_derep_T = 0.001   # /uM-sec
k_ind_T = 4.0       # /uM-sec
k_unin_T = 0.022    # /sec

# SmtB repression / induction
k_rep_S = 3.0       # /uM-sec
k_derep_S = 0.18    # /uM-sec
k_ind_S = 80        # /uM-sec
k_unin_S = 0.1      # /sec

# ---------------------------------------------------------------------------
# 2. Species definitions
# ---------------------------------------------------------------------------

# -- tetO module --
TetR = Species("TetR", material_type="protein")
D_tetO = Species("D_tetO", material_type="dna")
TetRD = Species("TetRD", material_type="complex")          # TetR:D_tetO (repressed)
tet = Species("tet", material_type="small_molecule")       # tetracycline inducer
TetR_tet = Species("TetR_tet", material_type="complex")    # TetR:tet (inactivated repressor)
RNAP = Species("RNAP", material_type="protein")
RD_tetO = Species("RD_tetO", material_type="complex")      # RNAP:D_tetO (open complex)
InvR_T = Species("InvR_T", material_type="rna")            # invader RNA produced from tetO

# -- smtO module --
SmtB = Species("SmtB", material_type="protein")
D_smtO = Species("D_smtO", material_type="dna")
SmtBD = Species("SmtBD", material_type="complex")          # SmtB:D_smtO (repressed)
Zn = Species("Zn", material_type="small_molecule")         # zinc inducer
SmtB_Zn = Species("SmtB_Zn", material_type="complex")      # SmtB:Zn (inactivated repressor)
RD_smtO = Species("RD_smtO", material_type="complex")      # RNAP:D_smtO (open complex)
InvR_S = Species("InvR_S", material_type="rna")            # invader RNA produced from smtO

# -- strand-displacement / AND-gate module --
AND_Gate = Species("AND_Gate", material_type="complex")    # dual-toehold AND gate strand
InvR_D = Species("InvR_D", material_type="rna")            # displaced strand from AND gate
Gated_TS = Species("Gated_TS", material_type="complex")    # byproduct duplex of AND-gate reaction
RQ = Species("RQ", material_type="complex")                # reporter:quencher complex
SD = Species("SD", material_type="complex")                # displaced strand from reporter
Q = Species("Q", material_type="rna")                      # freed quencher/output strand

# ---------------------------------------------------------------------------
# 3. Reactions
# ---------------------------------------------------------------------------

reactions = []

# ---- tetO-regulated template ----

# TetR + D_tetO <--> TetRD   (repression / de-repression)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO], outputs=[TetRD],
        k_forward=k_rep_T, k_reverse=k_derep_T
    )
)

# TetR + tet <--> TetR_tet   (induction / un-induction)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet], outputs=[TetR_tet],
        k_forward=k_ind_T, k_reverse=k_unin_T
    )
)

# RNAP + D_tetO <--> RD_tetO   (RNAP binding / unbinding)
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO], outputs=[RD_tetO],
        k_forward=k_bind, k_reverse=k_unbind
    )
)

# RD_tetO --> InvR_T + RNAP + D_tetO   (transcription, catalytic release)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO
    )
)

# ---- smtO-regulated template ----

# SmtB + D_smtO <--> SmtBD   (repression / de-repression)
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO], outputs=[SmtBD],
        k_forward=k_rep_S, k_reverse=k_derep_S
    )
)

# SmtB + Zn <--> SmtB_Zn   (induction / un-induction)
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn], outputs=[SmtB_Zn],
        k_forward=k_ind_S, k_reverse=k_unin_S
    )
)

# RNAP + D_smtO <--> RD_smtO   (RNAP binding / unbinding)
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO], outputs=[RD_smtO],
        k_forward=k_bind, k_reverse=k_unbind
    )
)

# RD_smtO --> InvR_S + RNAP + D_smtO   (transcription, catalytic release)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO
    )
)

# ---- Strand displacement AND-gate logic ----

# InvR_T + InvR_S + AND_Gate --> InvR_D + Gated_TS  (three-input AND-gate strand displacement)
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND
    )
)

# InvR_D + RQ --> SD + Q   (reporter strand displacement / signal release)
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ], outputs=[SD, Q],
        k_forward=k_SD
    )
)

# ---------------------------------------------------------------------------
# 4. Assemble the CRN
# ---------------------------------------------------------------------------

species = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

CRN = ChemicalReactionNetwork(species=species, reactions=reactions)

if __name__ == "__main__":
    print(CRN.pretty_print(show_rates=True, show_material=True))

    # Write out an SBML model of the CRN for use in a simulator
    # (e.g. COPASI, Tellurium, libRoadRunner).
    CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r3.xml')
    print("\nSBML model written to tetO_smtO_and_gate_crn.xml")

Species(N = 21) = {
small_molecule[tet] (@ 0),  small_molecule[Zn] (@ 0),  complex[TetR_tet] (@ 0),  complex[TetRD] (@ 0),  protein[TetR] (@ 0),  complex[SmtB_Zn] (@ 0),  complex[SmtBD] (@ 0),  protein[SmtB] (@ 0),  complex[SD] (@ 0),  complex[RQ] (@ 0),  protein[RNAP] (@ 0),  complex[RD_tetO] (@ 0),  complex[RD_smtO] (@ 0),  rna[Q] (@ 0),  rna[InvR_T] (@ 0),  rna[InvR_S] (@ 0),  rna[InvR_D] (@ 0),  complex[Gated_TS] (@ 0),  dna[D_tetO] (@ 0),  dna[D_smtO] (@ 0),  complex[AND_Gate] (@ 0),  
}

Reactions (10) = [
0. protein[TetR]+dna[D_tetO] <--> complex[TetRD]
 Kf=k_forward * protein_TetR * dna_D_tetO
 Kr=k_reverse * complex_TetRD
  k_forward=2.98
  k_reverse=0.001

1. protein[TetR]+small_molecule[tet] <--> complex[TetR_tet]
 Kf=k_forward * protein_TetR * small_molecule_tet
 Kr=k_reverse * complex_TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. protein[RNAP]+dna[D_tetO] <--> complex[RD_tetO]
 Kf=k_forward * protein_RNAP * dna_D_tetO
 Kr=k_reverse * complex_RD_tetO
  k_forward=56
  k_rev

### Attempt 4 [Successful]

In [25]:
"""
BioCRNpyler model of a tetO/smtO dual-input transcriptional AND gate
coupled to a downstream DNA strand-displacement (SD) reporter cascade.

Network summary
----------------
tetO-regulated template:
    TetR   + D_tetO  <-> TetRD              (k_rep_T   / k_derep_T)
    TetR   + tet     <-> TetR_tet           (k_ind_T   / k_unin_T)
    RNAP   + D_tetO  <-> RD_tetO            (k_bind    / k_unbind)
    RD_tetO          -> InvR_T + RNAP + D_tetO   (k_m_tetO, catalytic/irreversible)

smtO-regulated template:
    SmtB   + D_smtO  <-> SmtBD              (k_rep_S   / k_derep_S)
    SmtB   + Zn      <-> SmtB_Zn            (k_ind_S   / k_unin_S)
    RNAP   + D_smtO  <-> RD_smtO            (k_bind    / k_unbind)
    RD_smtO          -> InvR_S + RNAP + D_smtO   (k_m_smtO, catalytic/irreversible)

Strand-displacement layer:
    InvR_T + InvR_S + AND_Gate -> InvR_D + Gated_TS   (k_SD_AND, irreversible)
    InvR_D + RQ                -> SD + Q              (k_SD, irreversible)

All bimolecular rate constants are in 1/(uM*sec); all unimolecular rate
constants are in 1/sec, consistent with the parameter list supplied.
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ---------------------------------------------------------------------------
# 1. Rate constants (as given)
# ---------------------------------------------------------------------------
k_m_tetO   = 0.1      # /sec
k_m_smtO   = 0.04     # /sec
k_bind     = 56       # /uM-sec
k_unbind   = 0.2       # /sec
k_SD       = 0.04      # /uM-sec
k_SD_AND   = 0.08      # /uM-sec

k_rep_T    = 2.98      # /uM-sec
k_derep_T  = 0.001     # /uM-sec
k_ind_T    = 4.0       # /uM-sec
k_unin_T   = 0.022     # /sec

k_rep_S    = 3.0       # /uM-sec
k_derep_S  = 0.18      # /uM-sec
k_ind_S    = 80        # /uM-sec
k_unin_S   = 0.1       # /sec

# ---------------------------------------------------------------------------
# 2. Species
# ---------------------------------------------------------------------------
# -- tetO branch --
TetR      = Species("TetR",     material_type="protein")
D_tetO    = Species("D_tetO",   material_type="dna")
TetRD     = Species("TetRD",    material_type="complex")
tet       = Species("tet",      material_type="small_molecule")
TetR_tet  = Species("TetR_tet", material_type="complex")
RNAP      = Species("RNAP",     material_type="protein")
RD_tetO   = Species("RD_tetO",  material_type="complex")
InvR_T    = Species("InvR_T",   material_type="rna")

# -- smtO branch --
SmtB      = Species("SmtB",     material_type="protein")
D_smtO    = Species("D_smtO",   material_type="dna")
SmtBD     = Species("SmtBD",    material_type="complex")
Zn        = Species("Zn",       material_type="small_molecule")
SmtB_Zn   = Species("SmtB_Zn",  material_type="complex")
RD_smtO   = Species("RD_smtO",  material_type="complex")
InvR_S    = Species("InvR_S",   material_type="rna")

# -- strand-displacement / AND-gate layer --
AND_Gate  = Species("AND_Gate", material_type="dna")
InvR_D    = Species("InvR_D",   material_type="complex")
Gated_TS  = Species("Gated_TS", material_type="complex")
RQ        = Species("RQ",       material_type="complex")
SD        = Species("SD",       material_type="dna")
Q         = Species("Q",        material_type="small_molecule")

# ---------------------------------------------------------------------------
# 3. Reactions
# ---------------------------------------------------------------------------
reactions = []

# --- tetO-regulated template ---
reactions.append(Reaction.from_massaction(
    inputs=[TetR, D_tetO], outputs=[TetRD],
    k_forward=k_rep_T, k_reverse=k_derep_T))

reactions.append(Reaction.from_massaction(
    inputs=[TetR, tet], outputs=[TetR_tet],
    k_forward=k_ind_T, k_reverse=k_unin_T))

reactions.append(Reaction.from_massaction(
    inputs=[RNAP, D_tetO], outputs=[RD_tetO],
    k_forward=k_bind, k_reverse=k_unbind))

reactions.append(Reaction.from_massaction(
    inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO],
    k_forward=k_m_tetO))

# --- smtO-regulated template ---
reactions.append(Reaction.from_massaction(
    inputs=[SmtB, D_smtO], outputs=[SmtBD],
    k_forward=k_rep_S, k_reverse=k_derep_S))

reactions.append(Reaction.from_massaction(
    inputs=[SmtB, Zn], outputs=[SmtB_Zn],
    k_forward=k_ind_S, k_reverse=k_unin_S))

reactions.append(Reaction.from_massaction(
    inputs=[RNAP, D_smtO], outputs=[RD_smtO],
    k_forward=k_bind, k_reverse=k_unbind))

reactions.append(Reaction.from_massaction(
    inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO],
    k_forward=k_m_smtO))

# --- strand-displacement AND gate + reporter release ---
reactions.append(Reaction.from_massaction(
    inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS],
    k_forward=k_SD_AND))

reactions.append(Reaction.from_massaction(
    inputs=[InvR_D, RQ], outputs=[SD, Q],
    k_forward=k_SD))

# ---------------------------------------------------------------------------
# 4. Assemble the Chemical Reaction Network
# ---------------------------------------------------------------------------
species = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

CRN = ChemicalReactionNetwork(species=species, reactions=reactions)

if __name__ == "__main__":
    print(CRN.pretty_print(show_rates=True, show_material=True))

    # Write an SBML model that can be simulated with, e.g., bioscrape/COPASI/Tellurium
    CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r4.xml')
    print("\nSBML model written to tetO_smtO_and_gate_crn.xml")

Species(N = 21) = {
small_molecule[tet] (@ 0),  small_molecule[Zn] (@ 0),  complex[TetR_tet] (@ 0),  complex[TetRD] (@ 0),  protein[TetR] (@ 0),  complex[SmtB_Zn] (@ 0),  complex[SmtBD] (@ 0),  protein[SmtB] (@ 0),  dna[SD] (@ 0),  complex[RQ] (@ 0),  protein[RNAP] (@ 0),  complex[RD_tetO] (@ 0),  complex[RD_smtO] (@ 0),  small_molecule[Q] (@ 0),  rna[InvR_T] (@ 0),  rna[InvR_S] (@ 0),  complex[InvR_D] (@ 0),  complex[Gated_TS] (@ 0),  dna[D_tetO] (@ 0),  dna[D_smtO] (@ 0),  dna[AND_Gate] (@ 0),  
}

Reactions (10) = [
0. protein[TetR]+dna[D_tetO] <--> complex[TetRD]
 Kf=k_forward * protein_TetR * dna_D_tetO
 Kr=k_reverse * complex_TetRD
  k_forward=2.98
  k_reverse=0.001

1. protein[TetR]+small_molecule[tet] <--> complex[TetR_tet]
 Kf=k_forward * protein_TetR * small_molecule_tet
 Kr=k_reverse * complex_TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. protein[RNAP]+dna[D_tetO] <--> complex[RD_tetO]
 Kf=k_forward * protein_RNAP * dna_D_tetO
 Kr=k_reverse * complex_RD_tetO
  k_forward=56


### Attempt 5 [Successful]

In [26]:
"""
BioCRNpyler model of a tetO/smtO dual-input transcriptional AND gate
that produces an InvR-based strand-displacement signal.

Two independently regulated promoters (tetO and smtO) each transcribe an
"InvR" toehold-switch trigger RNA. Both trigger RNAs are required to open
an AND-gate strand-displacement complex, which in turn releases a reporter
by displacing a quencher strand from a reporter complex (RQ).

Run:
    python invR_and_gate_crn.py

This will print the CRN and write it out as an SBML file
(invR_and_gate_crn.xml) that can be loaded into COPASI, Tellurium,
BioSimulator, etc. for simulation.
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ---------------------------------------------------------------------------
# 1. Species
# ---------------------------------------------------------------------------

# --- tetO branch ---
TetR = Species("TetR")                 # TetR repressor protein
D_tetO = Species("D_tetO")             # free tetO-regulated DNA template
TetRD = Species("TetRD")               # TetR bound (repressed) template
tet = Species("tet")                   # anhydrotetracycline (inducer)
TetR_tet = Species("TetR_tet")         # TetR sequestered by inducer
RNAP = Species("RNAP")                 # RNA polymerase (shared pool)
RD_tetO = Species("RD_tetO")           # RNAP bound to free tetO template
InvR_T = Species("InvR_T")             # trigger RNA transcribed from tetO

# --- smtO branch ---
SmtB = Species("SmtB")                 # SmtB repressor protein
D_smtO = Species("D_smtO")             # free smtO-regulated DNA template
SmtBD = Species("SmtBD")               # SmtB bound (repressed) template
Zn = Species("Zn")                     # Zn(II) inducer
SmtB_Zn = Species("SmtB_Zn")           # SmtB sequestered by Zn(II)
RD_smtO = Species("RD_smtO")           # RNAP bound to free smtO template
InvR_S = Species("InvR_S")             # trigger RNA transcribed from smtO

# --- Strand-displacement / AND-gate logic ---
AND_Gate = Species("AND_Gate")         # dual-toehold AND-gate complex
InvR_D = Species("InvR_D")             # released downstream displacer strand
Gated_TS = Species("Gated_TS")         # spent (fully displaced) AND-gate complex
RQ = Species("RQ")                     # reporter:quencher complex
SD = Species("SD")                     # spent displacer:reporter waste complex
Q = Species("Q")                       # released quencher strand (readout)

species = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

# ---------------------------------------------------------------------------
# 2. Rate constants (all in uM / sec unless noted; unimolecular rates in 1/sec)
# ---------------------------------------------------------------------------

k_m_tetO = 0.1      # /sec   -- transcription (isomerization/elongation) from RD_tetO
k_m_smtO = 0.04      # /sec   -- transcription (isomerization/elongation) from RD_smtO

k_bind = 56.0        # /uM-sec -- RNAP binding to free promoter (shared by both templates)
k_unbind = 0.2        # /sec    -- RNAP unbinding from promoter (shared by both templates)

k_SD = 0.04       # /uM-sec -- InvR_D-driven strand displacement on RQ complex
k_SD_AND = 0.08       # /uM-sec -- dual-trigger strand displacement on the AND-gate

k_rep_T = 2.98      # /uM-sec -- TetR binding/repressing D_tetO
k_derep_T = 0.001     # /uM-sec -- TetR unbinding/derepressing D_tetO
k_ind_T = 4.0       # /uM-sec -- TetR sequestration by tet (induction)
k_unin_T = 0.022     # /sec    -- TetR release from TetR-tet complex

k_rep_S = 3.0       # /uM-sec -- SmtB binding/repressing D_smtO
k_derep_S = 0.18      # /uM-sec -- SmtB unbinding/derepressing D_smtO
k_ind_S = 80.0      # /uM-sec -- SmtB sequestration by Zn (induction)
k_unin_S = 0.1       # /sec    -- SmtB release from SmtB-Zn complex

# ---------------------------------------------------------------------------
# 3. Reactions
# ---------------------------------------------------------------------------

reactions = []

# ---- tetO-regulated template ----

# TetR + D_tetO <--> TetRD          (repression / derepression)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO], outputs=[TetRD],
        k_forward=k_rep_T, k_reverse=k_derep_T,
    )
)

# TetR + tet <--> TetR-tet          (induction / uninduction)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet], outputs=[TetR_tet],
        k_forward=k_ind_T, k_reverse=k_unin_T,
    )
)

# RNAP + D_tetO <--> RD_tetO        (RNAP binding / unbinding)
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO], outputs=[RD_tetO],
        k_forward=k_bind, k_reverse=k_unbind,
    )
)

# RD_tetO --> InvR_T + RNAP + D_tetO   (transcription, RNAP/template recycled)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO,
    )
)

# ---- smtO-regulated template ----

# SmtB + D_smtO <--> SmtBD          (repression / derepression)
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO], outputs=[SmtBD],
        k_forward=k_rep_S, k_reverse=k_derep_S,
    )
)

# SmtB + Zn <--> SmtB-Zn            (induction / uninduction)
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn], outputs=[SmtB_Zn],
        k_forward=k_ind_S, k_reverse=k_unin_S,
    )
)

# RNAP + D_smtO <--> RD_smtO        (RNAP binding / unbinding)
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO], outputs=[RD_smtO],
        k_forward=k_bind, k_reverse=k_unbind,
    )
)

# RD_smtO --> InvR_S + RNAP + D_smtO   (transcription, RNAP/template recycled)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO,
    )
)

# ---- Strand displacement / AND-gate logic ----

# InvR_T + InvR_S + AND-Gate --> InvR_D + Gated_TS  (both triggers required)
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND,
    )
)

# InvR_D + RQ --> SD + Q            (reporter strand displacement / readout)
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ], outputs=[SD, Q],
        k_forward=k_SD,
    )
)

# ---------------------------------------------------------------------------
# 4. Assemble and export the CRN
# ---------------------------------------------------------------------------

CRN = ChemicalReactionNetwork(species=species, reactions=reactions)

if __name__ == "__main__":
    print(CRN.pretty_print())
    CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r5.xml')
    print("\nSBML file written to invR_and_gate_crn.xml")

Species(N = 21) = {
tet (@ 0),  Zn (@ 0),  TetR_tet (@ 0),  TetRD (@ 0),  TetR (@ 0),  SmtB_Zn (@ 0),  SmtBD (@ 0),  SmtB (@ 0),  SD (@ 0),  RQ (@ 0),  RNAP (@ 0),  RD_tetO (@ 0),  RD_smtO (@ 0),  Q (@ 0),  InvR_T (@ 0),  InvR_S (@ 0),  InvR_D (@ 0),  Gated_TS (@ 0),  D_tetO (@ 0),  D_smtO (@ 0),  AND_Gate (@ 0),  
}

Reactions (10) = [
0. TetR+D_tetO <--> TetRD
 Kf=k_forward * TetR * D_tetO
 Kr=k_reverse * TetRD
  k_forward=2.98
  k_reverse=0.001

1. TetR+tet <--> TetR_tet
 Kf=k_forward * TetR * tet
 Kr=k_reverse * TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. RNAP+D_tetO <--> RD_tetO
 Kf=k_forward * RNAP * D_tetO
 Kr=k_reverse * RD_tetO
  k_forward=56.0
  k_reverse=0.2

3. RD_tetO --> InvR_T+RNAP+D_tetO
 Kf=k_forward * RD_tetO
  k_forward=0.1

4. SmtB+D_smtO <--> SmtBD
 Kf=k_forward * SmtB * D_smtO
 Kr=k_reverse * SmtBD
  k_forward=3.0
  k_reverse=0.18

5. SmtB+Zn <--> SmtB_Zn
 Kf=k_forward * SmtB * Zn
 Kr=k_reverse * SmtB_Zn
  k_forward=80.0
  k_reverse=0.1

6. RNAP+D_smtO <--> RD_

### Attempt 6 [Successful]

In [27]:
"""
BioCRNpyler script for a TetR/SmtB dual-input transcriptional
AND-gate network with downstream DNA strand-displacement readout.

Modules
-------
1. tetO-regulated template  (TetR repressor, tetracycline (tet) inducer)
2. smtO-regulated template  (SmtB repressor, Zn2+ inducer)
3. Strand-displacement cascade combining the two transcriptional
   outputs (InvR_T, InvR_S) into a final reporter release (Q).

All reactions are built as elementary mass-action steps using
biocrnpyler's low-level Species / Reaction / ChemicalReactionNetwork
API (no Mixture/Component machinery is needed for this fixed,
already-fully-specified reaction list).

Rate units: bimolecular rates are /uM/sec, unimolecular rates are /sec,
consistent with the parameter table supplied.
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ---------------------------------------------------------------------
# 1. Rate constants (uM, sec)
# ---------------------------------------------------------------------

# Catalytic / transcription rates
k_m_tetO = 0.1      # /sec   RD_tetO -> InvR_T + RNAP + D_tetO
k_m_smtO = 0.04      # /sec   RD_smtO -> InvR_S + RNAP + D_smtO

# RNAP <-> DNA (template) binding, shared by both templates
k_bind = 56.0       # /uM-sec
k_unbind = 0.2       # /sec

# Strand-displacement rates
k_SD = 0.04       # /uM-sec   InvR_D + RQ -> SD + Q
k_SD_AND = 0.08       # /uM-sec   InvR_T + InvR_S + AND-Gate -> InvR_D + Gated_TS

# TetR repressor module
k_rep_T = 2.98      # /uM-sec   TetR + D_tetO -> TetRD
k_derep_T = 0.001     # /uM-sec   TetRD -> TetR + D_tetO
k_ind_T = 4.0       # /uM-sec   TetR + tet -> TetR-tet
k_unin_T = 0.022     # /sec      TetR-tet -> TetR + tet

# SmtB repressor module
k_rep_S = 3.0       # /uM-sec   SmtB + D_smtO -> SmtBD
k_derep_S = 0.18      # /uM-sec   SmtBD -> SmtB + D_smtO
k_ind_S = 80.0      # /uM-sec   SmtB + Zn -> SmtB-Zn
k_unin_S = 0.1       # /sec      SmtB-Zn -> SmtB + Zn

# ---------------------------------------------------------------------
# 2. Species
# ---------------------------------------------------------------------

# --- tetO module ---
TetR = Species("TetR")
D_tetO = Species("D_tetO")
TetRD = Species("TetRD")
tet = Species("tet")
TetR_tet = Species("TetRtet")
RNAP = Species("RNAP")
RD_tetO = Species("RD_tetO")
InvR_T = Species("InvR_T")

# --- smtO module ---
SmtB = Species("SmtB")
D_smtO = Species("D_smtO")
SmtBD = Species("SmtBD")
Zn = Species("Zn")
SmtB_Zn = Species("SmtBZn")
RD_smtO = Species("RD_smtO")
InvR_S = Species("InvR_S")

# --- strand-displacement module ---
AND_Gate = Species("AND_Gate")
InvR_D = Species("InvR_D")
Gated_TS = Species("Gated_TS")
RQ = Species("RQ")
SD = Species("SD")
Q = Species("Q")

species = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

# ---------------------------------------------------------------------
# 3. Reactions
# ---------------------------------------------------------------------

reactions = []

# ---- tetO-regulated template ----

# TetR + D_tetO <--(k_rep_T, k_derep_T)--> TetRD
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=k_rep_T,
        k_reverse=k_derep_T,
    )
)

# TetR + tet <--(k_ind_T, k_unin_T)--> TetR-tet
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=k_ind_T,
        k_reverse=k_unin_T,
    )
)

# RNAP + D_tetO <--(k_bind, k_unbind)--> RD_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_tetO --(k_m_tetO)--> InvR_T + RNAP + D_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO,
    )
)

# ---- smtO-regulated template ----

# SmtB + D_smtO <--(k_rep_S, k_derep_S)--> SmtBD
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=k_rep_S,
        k_reverse=k_derep_S,
    )
)

# SmtB + Zn <--(k_ind_S, k_unin_S)--> SmtB-Zn
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=k_ind_S,
        k_reverse=k_unin_S,
    )
)

# RNAP + D_smtO <--(k_bind, k_unbind)--> RD_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_smtO --(k_m_smtO)--> InvR_S + RNAP + D_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO,
    )
)

# ---- strand displacement / AND logic ----

# InvR_T + InvR_S + AND-Gate --(k_SD_AND)--> InvR_D + Gated_TS
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND,
    )
)

# InvR_D + RQ --(k_SD)--> SD + Q
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=k_SD,
    )
)

# ---------------------------------------------------------------------
# 4. Initial concentrations (uM) -- edit as needed for your simulation
# ---------------------------------------------------------------------

initial_concentrations = {
    TetR: 0.1,
    D_tetO: 0.02,
    tet: 0.0,
    RNAP: 0.05,
    SmtB: 0.1,
    D_smtO: 0.02,
    Zn: 0.0,
    AND_Gate: 0.05,
    RQ: 0.1,
    # all other species (complexes, products) default to 0
}

# ---------------------------------------------------------------------
# 5. Assemble the CRN
# ---------------------------------------------------------------------

CRN = ChemicalReactionNetwork(
    species=species,
    reactions=reactions,
    initial_concentration_dict=initial_concentrations,
)

print(CRN.pretty_print(show_rates=True, show_keys=False))

if __name__ == "__main__":
    # Write out an SBML model that can be loaded into any standard
    # CRN/ODE simulator (COPASI, libRoadRunner, Tellurium, etc.)
    CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r6.xml')
    print("\nSBML model written to tetO_smtO_crn.xml")

Species(N = 21) = {
TetR (@ 0.1),  SmtB (@ 0.1),  RQ (@ 0.1),  RNAP (@ 0.05),  AND_Gate (@ 0.05),  D_tetO (@ 0.02),  D_smtO (@ 0.02),  tet (@ 0.0),  Zn (@ 0.0),  TetRtet (@ 0),  TetRD (@ 0),  SmtBZn (@ 0),  SmtBD (@ 0),  SD (@ 0),  RD_tetO (@ 0),  RD_smtO (@ 0),  Q (@ 0),  InvR_T (@ 0),  InvR_S (@ 0),  InvR_D (@ 0),  Gated_TS (@ 0),  
}

Reactions (10) = [
0. TetR+D_tetO <--> TetRD
 Kf=k_forward * TetR * D_tetO
 Kr=k_reverse * TetRD
  k_forward=2.98
  k_reverse=0.001

1. TetR+tet <--> TetRtet
 Kf=k_forward * TetR * tet
 Kr=k_reverse * TetRtet
  k_forward=4.0
  k_reverse=0.022

2. RNAP+D_tetO <--> RD_tetO
 Kf=k_forward * RNAP * D_tetO
 Kr=k_reverse * RD_tetO
  k_forward=56.0
  k_reverse=0.2

3. RD_tetO --> InvR_T+RNAP+D_tetO
 Kf=k_forward * RD_tetO
  k_forward=0.1

4. SmtB+D_smtO <--> SmtBD
 Kf=k_forward * SmtB * D_smtO
 Kr=k_reverse * SmtBD
  k_forward=3.0
  k_reverse=0.18

5. SmtB+Zn <--> SmtBZn
 Kf=k_forward * SmtB * Zn
 Kr=k_reverse * SmtBZn
  k_forward=80.0
  k_reverse=0.1

6. RNAP

### Attempt 7 [Successful]

In [28]:
"""
BioCRNpyler script for a two-input (tetracycline / zinc) transcriptional
AND-gate coupled to a downstream DNA strand-displacement (SD) reporter
cascade.

Two independent, RNAP-driven, repressor-regulated templates (tetO and
smtO) each produce a single-stranded "invader" species (InvR_T / InvR_S)
whenever their cognate repressor is inactivated by its small-molecule
inducer (tetracycline / Zn2+) and RNAP is bound. The two invaders are
combined on an AND-gate substrate by strand displacement to release a
third invader (InvR_D), which then displaces a fluorescent reporter
complex (RQ -> SD + Q).

All reactions are built directly as low-level mass-action Reaction
objects and assembled into a ChemicalReactionNetwork (CRN), which is
the appropriate BioCRNpyler abstraction when the network topology
(rather than a Mixture/Component library) is specified explicitly, as
here.
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ---------------------------------------------------------------------
# 1. Rate constants (converted to consistent BioCRNpyler units:
#    uM^-1 s^-1 for bimolecular reactions, s^-1 for unimolecular ones)
# ---------------------------------------------------------------------

# Transcription (mRNA/invader production) rates
k_m_tetO = 0.1      # /sec
k_m_smtO = 0.04      # /sec

# RNAP binding to either promoter (shared)
k_bind = 56.0        # /uM-sec
k_unbind = 0.2        # /sec

# Strand-displacement rates
k_SD = 0.04        # /uM-sec   (InvR_D + RQ -> SD + Q)
k_SD_AND = 0.08        # /uM-sec   (InvR_T + InvR_S + AND-Gate -> InvR_D + Gated_TS)

# tetO / TetR repression-derepression and induction-uninduction
k_rep_T = 2.98       # /uM-sec  (TetR + D_tetO -> TetRD)
k_derep_T = 0.001      # /uM-sec  (TetRD -> TetR + D_tetO)
k_ind_T = 4.0        # /uM-sec  (TetR + tet -> TetR-tet)
k_unin_T = 0.022      # /sec     (TetR-tet -> TetR + tet)

# smtO / SmtB repression-derepression and induction-uninduction
k_rep_S = 3.0        # /uM-sec  (SmtB + D_smtO -> SmtBD)
k_derep_S = 0.18       # /uM-sec  (SmtBD -> SmtB + D_smtO)
k_ind_S = 80.0       # /uM-sec  (SmtB + Zn -> SmtB-Zn)
k_unin_S = 0.1        # /sec     (SmtB-Zn -> SmtB + Zn)

# ---------------------------------------------------------------------
# 2. Species
# ---------------------------------------------------------------------

# --- tetO-regulated template ---
TetR = Species("TetR", material_type="protein")
D_tetO = Species("D_tetO", material_type="dna")
TetRD = Species("TetRD", material_type="complex")          # TetR bound to D_tetO (repressed)
tet = Species("tet", material_type="smallmol")            # tetracycline inducer
TetR_tet = Species("TetR_tet", material_type="complex")          # inactivated TetR-tetracycline complex
RNAP = Species("RNAP", material_type="protein")
RD_tetO = Species("RD_tetO", material_type="complex")          # RNAP bound to D_tetO (open complex)
InvR_T = Species("InvR_T", material_type="rna")              # invader strand from tetO template

# --- smtO-regulated template ---
SmtB = Species("SmtB", material_type="protein")
D_smtO = Species("D_smtO", material_type="dna")
SmtBD = Species("SmtBD", material_type="complex")          # SmtB bound to D_smtO (repressed)
Zn = Species("Zn", material_type="smallmol")            # zinc inducer
SmtB_Zn = Species("SmtB_Zn", material_type="complex")          # inactivated SmtB-Zn complex
RD_smtO = Species("RD_smtO", material_type="complex")          # RNAP bound to D_smtO (open complex)
InvR_S = Species("InvR_S", material_type="rna")              # invader strand from smtO template

# --- strand-displacement / AND-gate cascade ---
AND_Gate = Species("AND_Gate", material_type="dna")            # dual-toehold AND-gate substrate
InvR_D = Species("InvR_D", material_type="rna")              # invader released by the AND-gate
Gated_TS = Species("Gated_TS", material_type="complex")          # waste complex from AND-gate reaction
RQ = Species("RQ", material_type="complex")          # reporter complex (quenched)
SD = Species("SD", material_type="complex")          # displaced strand (waste)
Q = Species("Q", material_type="rna")               # freed fluorescent reporter strand

# ---------------------------------------------------------------------
# 3. Reactions
# ---------------------------------------------------------------------
reactions = []

# ---- tetO-regulated template ----

# TetR + D_tetO <-> TetRD   (repression / de-repression)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=k_rep_T,
        k_reverse=k_derep_T,
    )
)

# TetR + tet <-> TetR-tet   (induction / un-induction)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=k_ind_T,
        k_reverse=k_unin_T,
    )
)

# RNAP + D_tetO <-> RD_tetO   (RNAP binding / unbinding)
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_tetO -> InvR_T + RNAP + D_tetO   (transcription / invader production)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO,
    )
)

# ---- smtO-regulated template ----

# SmtB + D_smtO <-> SmtBD   (repression / de-repression)
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=k_rep_S,
        k_reverse=k_derep_S,
    )
)

# SmtB + Zn <-> SmtB-Zn   (induction / un-induction)
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=k_ind_S,
        k_reverse=k_unin_S,
    )
)

# RNAP + D_smtO <-> RD_smtO   (RNAP binding / unbinding)
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_smtO -> InvR_S + RNAP + D_smtO   (transcription / invader production)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO,
    )
)

# ---- strand-displacement cascade / AND-gate ----

# InvR_T + InvR_S + AND-Gate -> InvR_D + Gated_TS   (two-input AND gate)
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND,
    )
)

# InvR_D + RQ -> SD + Q   (reporter strand displacement / signal readout)
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=k_SD,
    )
)

# ---------------------------------------------------------------------
# 4. Assemble and export the CRN
# ---------------------------------------------------------------------
species = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

CRN = ChemicalReactionNetwork(species=species, reactions=reactions)

if __name__ == "__main__":
    print(CRN.pretty_print(show_rates=True, show_material=True))

    # Export as an SBML model for simulation in other tools (e.g. COPASI,
    # Tellurium) or as a BioCRNpyler .xml file for reloading later.
    CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r7.xml')
    print("\nSBML model written to tet_smt_and_gate_crn.xml")

Species(N = 21) = {
smallmol[tet] (@ 0),  smallmol[Zn] (@ 0),  complex[TetR_tet] (@ 0),  complex[TetRD] (@ 0),  protein[TetR] (@ 0),  complex[SmtB_Zn] (@ 0),  complex[SmtBD] (@ 0),  protein[SmtB] (@ 0),  complex[SD] (@ 0),  complex[RQ] (@ 0),  protein[RNAP] (@ 0),  complex[RD_tetO] (@ 0),  complex[RD_smtO] (@ 0),  rna[Q] (@ 0),  rna[InvR_T] (@ 0),  rna[InvR_S] (@ 0),  rna[InvR_D] (@ 0),  complex[Gated_TS] (@ 0),  dna[D_tetO] (@ 0),  dna[D_smtO] (@ 0),  dna[AND_Gate] (@ 0),  
}

Reactions (10) = [
0. protein[TetR]+dna[D_tetO] <--> complex[TetRD]
 Kf=k_forward * protein_TetR * dna_D_tetO
 Kr=k_reverse * complex_TetRD
  k_forward=2.98
  k_reverse=0.001

1. protein[TetR]+smallmol[tet] <--> complex[TetR_tet]
 Kf=k_forward * protein_TetR * smallmol_tet
 Kr=k_reverse * complex_TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. protein[RNAP]+dna[D_tetO] <--> complex[RD_tetO]
 Kf=k_forward * protein_RNAP * dna_D_tetO
 Kr=k_reverse * complex_RD_tetO
  k_forward=56.0
  k_reverse=0.2

3. complex[RD_te

### Attempt 8 [Successful]

In [29]:
"""
BioCRNpyler script for the InvR toehold-switch AND-gate CRN.

This builds the Chemical Reaction Network (CRN) described by:
  - the tetO-regulated transcriptional template (produces InvR_T)
  - the smtO-regulated transcriptional template (produces InvR_S)
  - the downstream strand-displacement (AND-gate) reactions

All reactions are constructed with explicit mass-action rate constants
using biocrnpyler's core Species / Reaction / ChemicalReactionNetwork
objects (no Mixture/Component abstraction is needed here since every
reaction and rate constant is fully specified by hand).
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ---------------------------------------------------------------------
# 1. Rate constants (converted to consistent units: uM and sec)
# ---------------------------------------------------------------------

# tetO-regulated template
k_rep_T    = 2.98    # /uM-sec   TetR + D_tetO   -> TetRD
k_derep_T  = 0.001   # /uM-sec   TetRD           -> TetR + D_tetO
k_ind_T    = 4.0     # /uM-sec   TetR + tet      -> TetR-tet
k_unin_T   = 0.022   # /sec      TetR-tet        -> TetR + tet
k_m_tetO   = 0.1      # /sec      RD_tetO -> InvR_T + RNAP + D_tetO

# smtO-regulated template
k_rep_S    = 3.0     # /uM-sec   SmtB + D_smtO   -> SmtBD
k_derep_S  = 0.18    # /uM-sec   SmtBD           -> SmtB + D_smtO
k_ind_S    = 80.0    # /uM-sec   SmtB + Zn       -> SmtB-Zn
k_unin_S   = 0.1     # /sec      SmtB-Zn         -> SmtB + Zn
k_m_smtO   = 0.04     # /sec      RD_smtO -> InvR_S + RNAP + D_smtO

# Shared RNAP-promoter binding (used identically by both templates)
k_bind     = 56.0    # /uM-sec   RNAP + D -> RD
k_unbind   = 0.2     # /sec      RD       -> RNAP + D

# Strand-displacement / AND-gate reactions
k_SD_AND   = 0.08    # /uM-sec   InvR_T + InvR_S + AND-Gate -> InvR_D + Gated_TS
k_SD       = 0.04    # /uM-sec   InvR_D + RQ -> SD + Q

# ---------------------------------------------------------------------
# 2. Species definitions
# ---------------------------------------------------------------------

# tetO template species
TetR       = Species("TetR",       material_type="protein")
D_tetO     = Species("D_tetO",     material_type="dna")
TetRD      = Species("TetRD",      material_type="complex")
tet        = Species("tet",        material_type="small_molecule")
TetR_tet   = Species("TetR_tet",   material_type="complex")
RNAP       = Species("RNAP",       material_type="protein")
RD_tetO    = Species("RD_tetO",    material_type="complex")
InvR_T     = Species("InvR_T",     material_type="rna")

# smtO template species
SmtB       = Species("SmtB",       material_type="protein")
D_smtO     = Species("D_smtO",     material_type="dna")
SmtBD      = Species("SmtBD",      material_type="complex")
Zn         = Species("Zn",         material_type="small_molecule")
SmtB_Zn    = Species("SmtB_Zn",    material_type="complex")
RD_smtO    = Species("RD_smtO",    material_type="complex")
InvR_S     = Species("InvR_S",     material_type="rna")

# Strand-displacement / logic-gate species
AND_Gate   = Species("AND_Gate",   material_type="complex")
InvR_D     = Species("InvR_D",     material_type="complex")
Gated_TS   = Species("Gated_TS",   material_type="complex")
RQ         = Species("RQ",         material_type="complex")
SD         = Species("SD",         material_type="complex")
Q          = Species("Q",          material_type="complex")

# ---------------------------------------------------------------------
# 3. Reactions
# ---------------------------------------------------------------------

reactions = []

# --- tetO-regulated template -------------------------------------------------

# TetR + D_tetO <--(k_rep_T, k_derep_T)--> TetRD
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=k_rep_T,
        k_reverse=k_derep_T,
    )
)

# TetR + tet <--(k_ind_T, k_unin_T)--> TetR-tet
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=k_ind_T,
        k_reverse=k_unin_T,
    )
)

# RNAP + D_tetO <--(k_bind, k_unbind)--> RD_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_tetO --(k_m_tetO)--> InvR_T + RNAP + D_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO,
    )
)

# --- smtO-regulated template --------------------------------------------------

# SmtB + D_smtO <--(k_rep_S, k_derep_S)--> SmtBD
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=k_rep_S,
        k_reverse=k_derep_S,
    )
)

# SmtB + Zn <--(k_ind_S, k_unin_S)--> SmtB-Zn
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=k_ind_S,
        k_reverse=k_unin_S,
    )
)

# RNAP + D_smtO <--(k_bind, k_unbind)--> RD_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_smtO --(k_m_smtO)--> InvR_S + RNAP + D_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO,
    )
)

# --- Strand displacement / AND-gate reactions --------------------------------

# InvR_T + InvR_S + AND-Gate --(k_SD_AND)--> InvR_D + Gated_TS
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND,
    )
)

# InvR_D + RQ --(k_SD)--> SD + Q
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=k_SD,
    )
)

# ---------------------------------------------------------------------
# 4. Assemble the Chemical Reaction Network
# ---------------------------------------------------------------------

species_list = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

CRN = ChemicalReactionNetwork(species=species_list, reactions=reactions)

if __name__ == "__main__":
    print("=" * 70)
    print("InvR AND-gate Chemical Reaction Network")
    print("=" * 70)
    print(CRN.pretty_print(show_rates=True, show_keys=False))

    # Write out an SBML model for simulation in COPASI, Tellurium, etc.
    sbml_path = "invr_crn.xml"
    CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r8.xml')
    print(f"\nSBML model written to: {sbml_path}")

InvR AND-gate Chemical Reaction Network
Species(N = 21) = {
small_molecule[tet] (@ 0),  small_molecule[Zn] (@ 0),  complex[TetR_tet] (@ 0),  complex[TetRD] (@ 0),  protein[TetR] (@ 0),  complex[SmtB_Zn] (@ 0),  complex[SmtBD] (@ 0),  protein[SmtB] (@ 0),  complex[SD] (@ 0),  complex[RQ] (@ 0),  protein[RNAP] (@ 0),  complex[RD_tetO] (@ 0),  complex[RD_smtO] (@ 0),  complex[Q] (@ 0),  rna[InvR_T] (@ 0),  rna[InvR_S] (@ 0),  complex[InvR_D] (@ 0),  complex[Gated_TS] (@ 0),  dna[D_tetO] (@ 0),  dna[D_smtO] (@ 0),  complex[AND_Gate] (@ 0),  
}

Reactions (10) = [
0. protein[TetR]+dna[D_tetO] <--> complex[TetRD]
 Kf=k_forward * protein_TetR * dna_D_tetO
 Kr=k_reverse * complex_TetRD
  k_forward=2.98
  k_reverse=0.001

1. protein[TetR]+small_molecule[tet] <--> complex[TetR_tet]
 Kf=k_forward * protein_TetR * small_molecule_tet
 Kr=k_reverse * complex_TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. protein[RNAP]+dna[D_tetO] <--> complex[RD_tetO]
 Kf=k_forward * protein_RNAP * dna_D_tetO
 Kr=k_

### Attempt 9 [Successful]

In [30]:
"""
BioCRNpyler script: tetO/smtO dual-input genetic AND-gate CRN
================================================================

This script builds a Chemical Reaction Network (CRN) directly from
Species and mass-action Reaction objects (rather than using the
Mixture/Component abstraction), since the system described is a
custom, hand-specified set of reactions and rate constants rather
than a standard promoter/RBS/CDS assembly.

Reaction groups
----------------
1. tetO-regulated template  (TetR repression / induction, RNAP binding, transcription)
2. smtO-regulated template  (SmtB repression / induction, RNAP binding, transcription)
3. Strand-displacement (DNA) logic: AND-gate combination of the two
   transcriptional outputs (InvR_T, InvR_S) and downstream reporter
   strand displacement.

All bimolecular rate constants are given in 1/(uM*s) and unimolecular
rate constants in 1/s, matching the units in the prompt. BioCRNpyler
does not enforce units -- it simply uses whatever numeric values are
supplied for the mass-action propensities -- so the values below are
entered directly as given.
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# ---------------------------------------------------------------------
# 1. Rate constants (all in uM, sec units as specified)
# ---------------------------------------------------------------------

# Transcription (catalytic / turnover) rates
k_m_tetO = 0.1      # /sec
k_m_smtO = 0.04      # /sec

# RNAP binding to either promoter/template (shared for both templates)
k_bind = 56.0        # /uM-sec
k_unbind = 0.2        # /sec

# Strand displacement rates
k_SD = 0.04          # /uM-sec        (InvR_D + RQ -> SD + Q)
k_SD_AND = 0.08          # /uM-sec        (InvR_T + InvR_S + AND-Gate -> InvR_D + Gated_TS)

# TetR repressor / inducer kinetics
k_rep_T = 2.98         # /uM-sec   TetR + D_tetO -> TetRD
k_derep_T = 0.001        # /sec      TetRD -> TetR + D_tetO
k_ind_T = 4.0          # /uM-sec   TetR + tet -> TetR-tet
k_unin_T = 0.022        # /sec      TetR-tet -> TetR + tet

# SmtB repressor / inducer kinetics
k_rep_S = 3.0          # /uM-sec   SmtB + D_smtO -> SmtBD
k_derep_S = 0.18         # /sec      SmtBD -> SmtB + D_smtO
k_ind_S = 80.0         # /uM-sec   SmtB + Zn -> SmtB-Zn
k_unin_S = 0.1          # /sec      SmtB-Zn -> SmtB + Zn

# ---------------------------------------------------------------------
# 2. Species definitions
# ---------------------------------------------------------------------

# --- tetO-regulated template species ---
TetR       = Species("TetR",       material_type="protein")
D_tetO     = Species("D_tetO",     material_type="dna")
TetRD      = Species("TetRD",      material_type="complex")
tet        = Species("tet",        material_type="small_molecule")
TetR_tet   = Species("TetR_tet",   material_type="complex")
RNAP       = Species("RNAP",       material_type="protein")
RD_tetO    = Species("RD_tetO",    material_type="complex")
InvR_T     = Species("InvR_T",     material_type="rna")

# --- smtO-regulated template species ---
SmtB       = Species("SmtB",       material_type="protein")
D_smtO     = Species("D_smtO",     material_type="dna")
SmtBD      = Species("SmtBD",      material_type="complex")
Zn         = Species("Zn",         material_type="small_molecule")
SmtB_Zn    = Species("SmtB_Zn",    material_type="complex")
RD_smtO    = Species("RD_smtO",    material_type="complex")
InvR_S     = Species("InvR_S",     material_type="rna")

# --- strand-displacement logic species ---
AND_Gate   = Species("AND_Gate",   material_type="dna")
InvR_D     = Species("InvR_D",     material_type="rna")
Gated_TS   = Species("Gated_TS",   material_type="complex")
RQ         = Species("RQ",         material_type="complex")
SD         = Species("SD",         material_type="rna")
Q          = Species("Q",          material_type="complex")

species_list = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

# ---------------------------------------------------------------------
# 3. Reactions
# ---------------------------------------------------------------------

reactions = []

# ===== tetO-regulated template =====

# TetR + D_tetO <--(k_rep,T / k_derep,T)--> TetRD   (repression / de-repression)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO], outputs=[TetRD],
        k_forward=k_rep_T, k_reverse=k_derep_T,
    )
)

# TetR + tet <--(k_ind,T / k_unin,T)--> TetR-tet   (induction / un-induction)
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet], outputs=[TetR_tet],
        k_forward=k_ind_T, k_reverse=k_unin_T,
    )
)

# RNAP + D_tetO <--(k_bind / k_unbind)--> RD_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO], outputs=[RD_tetO],
        k_forward=k_bind, k_reverse=k_unbind,
    )
)

# RD_tetO --(k_m,tetO)--> InvR_T + RNAP + D_tetO   (transcription, catalytic/irreversible)
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO], outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO,
    )
)

# ===== smtO-regulated template =====

# SmtB + D_smtO <--(k_rep,S / k_derep,S)--> SmtBD
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO], outputs=[SmtBD],
        k_forward=k_rep_S, k_reverse=k_derep_S,
    )
)

# SmtB + Zn <--(k_ind,S / k_unin,S)--> SmtB-Zn
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn], outputs=[SmtB_Zn],
        k_forward=k_ind_S, k_reverse=k_unin_S,
    )
)

# RNAP + D_smtO <--(k_bind / k_unbind)--> RD_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO], outputs=[RD_smtO],
        k_forward=k_bind, k_reverse=k_unbind,
    )
)

# RD_smtO --(k_m,smtO)--> InvR_S + RNAP + D_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO], outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO,
    )
)

# ===== Strand displacement / AND-gate logic =====

# InvR_T + InvR_S + AND-Gate --(k_SD,AND)--> InvR_D + Gated_TS
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate], outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND,
    )
)

# InvR_D + RQ --(k_SD)--> SD + Q
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ], outputs=[SD, Q],
        k_forward=k_SD,
    )
)

# ---------------------------------------------------------------------
# 4. Assemble and export the CRN
# ---------------------------------------------------------------------

CRN = ChemicalReactionNetwork(species=species_list, reactions=reactions)

if __name__ == "__main__":
    print(CRN.pretty_print(show_rates=True, show_keys=False))

    # Write SBML for simulation in COPASI, Tellurium, libRoadRunner, etc.
    sbml_path = "tetO_smtO_CRN.xml"
    CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r9.xml')
    print(f"\nSBML model written to: {sbml_path}")

Species(N = 21) = {
small_molecule[tet] (@ 0),  small_molecule[Zn] (@ 0),  complex[TetR_tet] (@ 0),  complex[TetRD] (@ 0),  protein[TetR] (@ 0),  complex[SmtB_Zn] (@ 0),  complex[SmtBD] (@ 0),  protein[SmtB] (@ 0),  rna[SD] (@ 0),  complex[RQ] (@ 0),  protein[RNAP] (@ 0),  complex[RD_tetO] (@ 0),  complex[RD_smtO] (@ 0),  complex[Q] (@ 0),  rna[InvR_T] (@ 0),  rna[InvR_S] (@ 0),  rna[InvR_D] (@ 0),  complex[Gated_TS] (@ 0),  dna[D_tetO] (@ 0),  dna[D_smtO] (@ 0),  dna[AND_Gate] (@ 0),  
}

Reactions (10) = [
0. protein[TetR]+dna[D_tetO] <--> complex[TetRD]
 Kf=k_forward * protein_TetR * dna_D_tetO
 Kr=k_reverse * complex_TetRD
  k_forward=2.98
  k_reverse=0.001

1. protein[TetR]+small_molecule[tet] <--> complex[TetR_tet]
 Kf=k_forward * protein_TetR * small_molecule_tet
 Kr=k_reverse * complex_TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. protein[RNAP]+dna[D_tetO] <--> complex[RD_tetO]
 Kf=k_forward * protein_RNAP * dna_D_tetO
 Kr=k_reverse * complex_RD_tetO
  k_forward=56.0
  k_rever

### Attempt 10 [Successful]

In [31]:
"""
BioCRNpyler script: Two-input AND-gate genetic circuit
tetO-regulated and smtO-regulated templates driving a
strand-displacement AND-gate reporter circuit.

Uses BioCRNpyler's core Species / Reaction / ChemicalReactionNetwork
classes to directly encode the given mass-action reactions.
"""

from biocrnpyler import Species, Reaction, ChemicalReactionNetwork

# -----------------------------------------------------------------
# 1. PARAMETERS (all in consistent uM / sec units)
# -----------------------------------------------------------------

# tetO-regulated template
k_rep_T    = 2.98    # /uM-sec   TetR + D_tetO -> TetRD
k_derep_T  = 0.001    # /uM-sec   TetRD -> TetR + D_tetO
k_ind_T    = 4.0      # /uM-sec   TetR + tet -> TetR-tet
k_unin_T   = 0.022    # /sec      TetR-tet -> TetR + tet
k_m_tetO   = 0.1      # /sec      RD_tetO -> InvR_T + RNAP + D_tetO

# smtO-regulated template
k_rep_S    = 3.0      # /uM-sec   SmtB + D_smtO -> SmtBD
k_derep_S  = 0.18     # /uM-sec   SmtBD -> SmtB + D_smtO
k_ind_S    = 80.0     # /uM-sec   SmtB + Zn -> SmtB-Zn
k_unin_S   = 0.1       # /sec      SmtB-Zn -> SmtB + Zn
k_m_smtO   = 0.04     # /sec      RD_smtO -> InvR_S + RNAP + D_smtO

# Shared RNAP binding (used by both templates)
k_bind     = 56.0     # /uM-sec   RNAP + D -> RD
k_unbind   = 0.2      # /sec      RD -> RNAP + D

# Strand displacement / AND-gate
k_SD_AND   = 0.08     # /uM-sec   InvR_T + InvR_S + AND-Gate -> InvR_D + Gated_TS
k_SD       = 0.04     # /uM-sec   InvR_D + RQ -> SD + Q

# -----------------------------------------------------------------
# 2. SPECIES
# -----------------------------------------------------------------

# tetO branch
TetR      = Species("TetR", material_type="protein")
D_tetO    = Species("D_tetO", material_type="dna")
TetRD     = Species("TetRD", material_type="complex")
tet       = Species("tet", material_type="small_molecule")
TetR_tet  = Species("TetR_tet", material_type="complex")
RNAP      = Species("RNAP", material_type="protein")
RD_tetO   = Species("RD_tetO", material_type="complex")
InvR_T    = Species("InvR_T", material_type="rna")

# smtO branch
SmtB      = Species("SmtB", material_type="protein")
D_smtO    = Species("D_smtO", material_type="dna")
SmtBD     = Species("SmtBD", material_type="complex")
Zn        = Species("Zn", material_type="small_molecule")
SmtB_Zn   = Species("SmtB_Zn", material_type="complex")
RD_smtO   = Species("RD_smtO", material_type="complex")
InvR_S    = Species("InvR_S", material_type="rna")

# Strand-displacement / AND-gate layer
AND_Gate  = Species("AND_Gate", material_type="dna")
InvR_D    = Species("InvR_D", material_type="rna")
Gated_TS  = Species("Gated_TS", material_type="complex")
RQ        = Species("RQ", material_type="complex")
SD        = Species("SD", material_type="dna")
Q         = Species("Q", material_type="dna")

# -----------------------------------------------------------------
# 3. REACTIONS
# -----------------------------------------------------------------

reactions = []

# ---- tetO-regulated template ----

# TetR + D_tetO <--(k_rep,T, k_derep,T)--> TetRD
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, D_tetO],
        outputs=[TetRD],
        k_forward=k_rep_T,
        k_reverse=k_derep_T,
    )
)

# TetR + tet <--(k_ind,T, k_unin,T)--> TetR-tet
reactions.append(
    Reaction.from_massaction(
        inputs=[TetR, tet],
        outputs=[TetR_tet],
        k_forward=k_ind_T,
        k_reverse=k_unin_T,
    )
)

# RNAP + D_tetO <--(k_bind, k_unbind)--> RD_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_tetO],
        outputs=[RD_tetO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_tetO --(k_m,tetO)--> InvR_T + RNAP + D_tetO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_tetO],
        outputs=[InvR_T, RNAP, D_tetO],
        k_forward=k_m_tetO,
    )
)

# ---- smtO-regulated template ----

# SmtB + D_smtO <--(k_rep,S, k_derep,S)--> SmtBD
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, D_smtO],
        outputs=[SmtBD],
        k_forward=k_rep_S,
        k_reverse=k_derep_S,
    )
)

# SmtB + Zn <--(k_ind,S, k_unin,S)--> SmtB-Zn
reactions.append(
    Reaction.from_massaction(
        inputs=[SmtB, Zn],
        outputs=[SmtB_Zn],
        k_forward=k_ind_S,
        k_reverse=k_unin_S,
    )
)

# RNAP + D_smtO <--(k_bind, k_unbind)--> RD_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RNAP, D_smtO],
        outputs=[RD_smtO],
        k_forward=k_bind,
        k_reverse=k_unbind,
    )
)

# RD_smtO --(k_m,smtO)--> InvR_S + RNAP + D_smtO
reactions.append(
    Reaction.from_massaction(
        inputs=[RD_smtO],
        outputs=[InvR_S, RNAP, D_smtO],
        k_forward=k_m_smtO,
    )
)

# ---- Strand displacements / AND-gate ----

# InvR_T + InvR_S + AND-Gate --(k_SD,AND)--> InvR_D + Gated_TS
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_T, InvR_S, AND_Gate],
        outputs=[InvR_D, Gated_TS],
        k_forward=k_SD_AND,
    )
)

# InvR_D + RQ --(k_SD)--> SD + Q
reactions.append(
    Reaction.from_massaction(
        inputs=[InvR_D, RQ],
        outputs=[SD, Q],
        k_forward=k_SD,
    )
)

# -----------------------------------------------------------------
# 4. BUILD THE CHEMICAL REACTION NETWORK
# -----------------------------------------------------------------

species_list = [
    TetR, D_tetO, TetRD, tet, TetR_tet, RNAP, RD_tetO, InvR_T,
    SmtB, D_smtO, SmtBD, Zn, SmtB_Zn, RD_smtO, InvR_S,
    AND_Gate, InvR_D, Gated_TS, RQ, SD, Q,
]

CRN = ChemicalReactionNetwork(species=species_list, reactions=reactions)

print(CRN.pretty_print(show_rates=True, show_keys=False))

# -----------------------------------------------------------------
# 5. WRITE SBML / SIMULATE (optional)
# -----------------------------------------------------------------

CRN.write_sbml_file(f'{model_dir}/AND_gate_CRN_claude_r10.xml') 

# Example deterministic simulation using bioscrape (if installed)
if __name__ == "__main__":
    try:
        import numpy as np

        timepoints = np.linspace(0, 500, 1000)

        # Initial concentrations (uM) — edit as needed for your scenario
        x0 = {
            TetR.name: 0.5,
            D_tetO.name: 0.01,
            tet.name: 0.0,       # set >0 to induce TetR
            RNAP.name: 2.0,
            SmtB.name: 0.5,
            D_smtO.name: 0.01,
            Zn.name: 0.0,        # set >0 to induce SmtB
            AND_Gate.name: 0.05,
            RQ.name: 0.1,
        }

        result = CRN.simulate_with_bioscrape_deterministic(
            timepoints=timepoints, initial_condition_dict=x0
        )
        print(result.head())

    except ImportError:
        print("bioscrape not installed; skipping simulation. "
              "SBML file 'AND_gate_circuit.xml' was still generated.")

Species(N = 21) = {
small_molecule[tet] (@ 0),  small_molecule[Zn] (@ 0),  complex[TetR_tet] (@ 0),  complex[TetRD] (@ 0),  protein[TetR] (@ 0),  complex[SmtB_Zn] (@ 0),  complex[SmtBD] (@ 0),  protein[SmtB] (@ 0),  dna[SD] (@ 0),  complex[RQ] (@ 0),  protein[RNAP] (@ 0),  complex[RD_tetO] (@ 0),  complex[RD_smtO] (@ 0),  dna[Q] (@ 0),  rna[InvR_T] (@ 0),  rna[InvR_S] (@ 0),  rna[InvR_D] (@ 0),  complex[Gated_TS] (@ 0),  dna[D_tetO] (@ 0),  dna[D_smtO] (@ 0),  dna[AND_Gate] (@ 0),  
}

Reactions (10) = [
0. protein[TetR]+dna[D_tetO] <--> complex[TetRD]
 Kf=k_forward * protein_TetR * dna_D_tetO
 Kr=k_reverse * complex_TetRD
  k_forward=2.98
  k_reverse=0.001

1. protein[TetR]+small_molecule[tet] <--> complex[TetR_tet]
 Kf=k_forward * protein_TetR * small_molecule_tet
 Kr=k_reverse * complex_TetR_tet
  k_forward=4.0
  k_reverse=0.022

2. protein[RNAP]+dna[D_tetO] <--> complex[RD_tetO]
 Kf=k_forward * protein_RNAP * dna_D_tetO
 Kr=k_reverse * complex_RD_tetO
  k_forward=56.0
  k_reverse=0

AttributeError: 'ChemicalReactionNetwork' object has no attribute 'simulate_with_bioscrape_deterministic'

# Computing environment

In [32]:
%load_ext watermark
%watermark -v -p bioscrape,bokeh,jupyterlab,biocrnpyler

Python implementation: CPython
Python version       : 3.12.12
IPython version      : 9.12.0

bioscrape  : 1.2.1
bokeh      : 3.9.1
jupyterlab : 4.5.0
biocrnpyler: 1.3.2

